In [26]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [27]:
# CELL 1: Install dependencies (no Flash Attention — compile issues on Colab)
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl
!pip install -q pillow-avif-plugin num2words

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [28]:
# CELL 2: Optional HF login (speeds up downloads, avoids rate limits)
# Uncomment the next two lines if you have a token
from huggingface_hub import login
login()

In [29]:
# CELL 3: Load plantvillage-full and inspect structure
from datasets import load_dataset
from collections import Counter

ds = load_dataset("geraldmc/plantvillage-full", revision="v0.1.0")
train_ds = ds["train"]

print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)
print(f"Columns: {train_ds.column_names}")
print(f"Total samples: {len(train_ds)}")

ex = train_ds[0]
print("\nFirst sample fields:")
for k, v in ex.items():
    if k == "image":
        print(f"  {k}: PIL Image {v.size}")
    else:
        print(f"  {k}: {v}")

label_counts = Counter(train_ds["class_label"])
print(f"\nTotal classes: {len(label_counts)}")
print(f"Min samples/class: {min(label_counts.values())}")
print(f"Max samples/class: {max(label_counts.values())}")

DATASET STRUCTURE
Columns: ['image', 'class_label', 'class_idx', 'host', 'disease', 'split', 'leaf_id', 'leaf_grouped']
Total samples: 54304

First sample fields:
  image: PIL Image (256, 256)
  class_label: Apple___Apple_scab
  class_idx: 0
  host: Apple
  disease: Apple scab
  split: test
  leaf_id: Apple___Apple_scab::leaf_79
  leaf_grouped: True

Total classes: 38
Min samples/class: 152
Max samples/class: 5507


In [30]:
# CELL 4: Convert to ChatML format — keeps ONLY `image` and `messages` columns
def build_conversational_batch(batch):
    messages_list = []
    for label in batch["class_label"]:
        if "___" in label:
            host, disease = label.split("___", 1)
            disease_display = disease.replace("_", " ").strip()
        else:
            host, disease_display = "Unknown", label.replace("_", " ")

        if disease_display.lower() == "healthy":
            answer = f"**Plant:** {host}\n**Disease:** Healthy\n**Symptoms:** No visible disease symptoms."
        else:
            answer = f"**Plant:** {host}\n**Disease:** {disease_display}\n**Symptoms:** Visible symptoms consistent with {disease_display}."

        messages_list.append([
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": "Identify the plant disease in this image."}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": answer}]}
        ])

    return {
        "messages": messages_list,
        "image": batch["image"],
    }

print("Converting to conversational format...")
converted = train_ds.map(
    build_conversational_batch,
    batched=True,
    batch_size=256,
    remove_columns=train_ds.column_names,
    desc="Formatting",
    load_from_cache_file=False,
)

print(f"Converted: {len(converted)}")
print(f"Columns: {converted.column_names}")

s = converted[0]
print(f"\nimage type: {type(s['image'])}")
print(f"image size: {s['image'].size}")
print(f"messages turns: {len(s['messages'])}")
print(f"\nFirst sample messages:\n{s['messages']}")

Converting to conversational format...


Formatting:   0%|          | 0/54304 [00:00<?, ? examples/s]

Converted: 54304
Columns: ['image', 'messages']

image type: <class 'PIL.JpegImagePlugin.JpegImageFile'>
image size: (256, 256)
messages turns: 2

First sample messages:
[{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': 'Identify the plant disease in this image.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '**Plant:** Apple\n**Disease:** Apple scab\n**Symptoms:** Visible symptoms consistent with Apple scab.'}]}]


In [44]:
# CELL 5 (IMPROVED): Balanced subset — 120/class for better accuracy
import random
from collections import defaultdict

random.seed(43)   # different seed for better class coverage

class_indices = defaultdict(list)
for i, sample in enumerate(converted):
    answer = sample["messages"][1]["content"][0]["text"]
    lines = answer.split("\n")
    plant = lines[0].replace("**Plant:** ", "").strip()
    disease = lines[1].replace("**Disease:** ", "").strip()
    class_indices[f"{plant}___{disease}"].append(i)

MAX_PER_CLASS = 120     # ← doubled from 60
selected = []
for cls, idxs in class_indices.items():
    random.shuffle(idxs)
    selected.extend(idxs[:MAX_PER_CLASS])

random.shuffle(selected)
subset = converted.select(selected)

split = subset.train_test_split(test_size=0.1, seed=43)
train_data = split["train"]
val_data = split["test"]

print(f"Subset: {len(subset)} | Train: {len(train_data)} | Val: {len(val_data)}")
print(f"Classes: {len(class_indices)}")
print(f"Steps/epoch (batch=1, accum=8): {len(train_data) // 8}")
print(f"Estimated training time: ~{len(train_data) // 8 * 2 * 8 / 60:.0f} min total (2 epochs)")

Subset: 4560 | Train: 4104 | Val: 456
Classes: 38
Steps/epoch (batch=1, accum=8): 513
Estimated training time: ~137 min total (2 epochs)


In [45]:
# CELL 6: Load SmolVLM-500M with 4-bit quantization + LoRA
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Processor with reduced image resolution (biggest speed win)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    size={"longest_edge": 512},   # default is 2048 → ~16× fewer vision tokens
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading model with 4-bit quantization...")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
    init_lora_weights="gaussian",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model loaded successfully.")

Loading model with 4-bit quantization...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

trainable params: 9,568,256 || all params: 517,050,560 || trainable%: 1.8505
Model loaded successfully.


In [47]:
# CELL 7 (IMPROVED): Train 2 epochs for ~85% accuracy
from trl import SFTConfig, SFTTrainer
from PIL import Image

def collate_fn(examples):
    texts = []
    images = []
    for ex in examples:
        text = processor.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
        img = ex.get("image", ex.get("images"))
        if isinstance(img, list):
            img = img[0]
        if not isinstance(img, Image.Image):
            raise TypeError(f"Expected PIL Image, got {type(img)}")
        images.append(img)

    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=768,
    )
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    return batch

per_device_batch = 1
grad_accum = 8
num_epochs = 2                    # ← doubled from 1
effective_batch = per_device_batch * grad_accum

steps_per_epoch = len(train_data) // effective_batch
total_steps = steps_per_epoch * num_epochs
warmup_steps = max(1, int(total_steps * 0.03))

print(f"Steps/epoch: {steps_per_epoch} | Total: {total_steps} | Warmup: {warmup_steps}")
print(f"Estimated training time: ~{total_steps * 8 / 60:.0f} min")

training_args = SFTConfig(
    output_dir="./smolvlm-plantvillage-output-v2",
    num_train_epochs=num_epochs,
    per_device_train_batch_size=per_device_batch,
    gradient_accumulation_steps=grad_accum,
    learning_rate=1.5e-4,          # ← lowered from 2e-4 for 2-epoch stability
    warmup_steps=warmup_steps,
    logging_steps=10,
    save_strategy="no",
    eval_strategy="no",
    gradient_checkpointing=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    max_length=768,
    loss_type="nll",
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=None,
    data_collator=collate_fn,
    processing_class=processor,
)

print("Starting 2-epoch training...")
trainer.train()

adapter_dir = "/kaggle/working/smolvlm-plantvillage-lora-v2"
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
print(f"Adapter saved to {adapter_dir}")

Steps/epoch: 513 | Total: 1026 | Warmup: 30
Estimated training time: ~137 min
Starting 2-epoch training...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,8.599206
20,4.163654
30,0.712010
40,0.261756
50,0.164540
60,0.086032
70,0.070224
80,0.046721
90,0.043146
100,0.031364


Adapter saved to /kaggle/working/smolvlm-plantvillage-lora-v2


In [48]:
# CELL 8: Quick inference check on a validation sample
import torch

test_sample = val_data[0]
test_image = test_sample["image"]

messages = [{
    "role": "user",
    "content": [
        {"type": "image"},
        {"type": "text", "text": "Identify the plant disease in this image."}
    ]
}]

model.eval()
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[test_image], return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)

response = processor.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=" * 60)
print("MODEL OUTPUT")
print("=" * 60)
print(response)
print("\n" + "=" * 60)
print("GROUND TRUTH")
print("=" * 60)
print(test_sample["messages"][1]["content"][0]["text"])

MODEL OUTPUT
 **Plant:** Tomato
**Disease:** Tomato Yellow Leaf Curl Virus
**Symptoms:** Visible symptoms consistent with Tomato Yellow Leaf Curl Virus.

GROUND TRUTH
**Plant:** Tomato
**Disease:** Tomato Yellow Leaf Curl Virus
**Symptoms:** Visible symptoms consistent with Tomato Yellow Leaf Curl Virus.


In [49]:
# CELL 8A: Complete accuracy metrics — accuracy, F1, precision, recall, confusion matrix
from PIL import Image
import torch
import numpy as np
from tqdm import tqdm
from collections import Counter, defaultdict
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
import json

# ---- Helper: extract disease name from model output ----
def get_disease(text):
    """Extract disease name from **Disease:** line."""
    for line in text.split("\n"):
        if "Disease:" in line:
            return line.replace("**Disease:**", "").replace("*", "").strip()
    return "unknown"

def get_plant(text):
    """Extract plant name from **Plant:** line."""
    for line in text.split("\n"):
        if "Plant:" in line:
            return line.replace("**Plant:**", "").replace("*", "").strip()
    return "unknown"

# ---- Run evaluation ----
N_EVAL = min(200, len(val_data))   # 200 samples for stable metrics; use full val if time permits
model.eval()

y_true_disease = []
y_pred_disease = []
y_true_plant = []
y_pred_plant = []
full_examples = []   # store (gt_text, pred_text, image_idx) for qualitative review

print(f"Evaluating {N_EVAL} validation samples...")
print("This takes ~4-5 seconds per sample on T4, ~15 minutes total.\n")

for i in tqdm(range(N_EVAL), desc="Evaluating"):
    sample = val_data[i]
    img = sample["image"]
    gt_text = sample["messages"][1]["content"][0]["text"]

    msgs = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Identify the plant disease in this image."}
    ]}]

    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[img], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, do_sample=False)

    pred_text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    gt_dis = get_disease(gt_text).lower()
    pr_dis = get_disease(pred_text).lower()
    gt_pl = get_plant(gt_text).lower()
    pr_pl = get_plant(pred_text).lower()

    y_true_disease.append(gt_dis)
    y_pred_disease.append(pr_dis)
    y_true_plant.append(gt_pl)
    y_pred_plant.append(pr_pl)

    full_examples.append({
        "idx": i,
        "gt_text": gt_text,
        "pred_text": pred_text,
        "gt_disease": gt_dis,
        "pred_disease": pr_dis,
        "correct": gt_dis == pr_dis,
    })

print("\n" + "=" * 70)
print("DISEASE-LEVEL METRICS")
print("=" * 70)

# ---- Accuracy ----
acc = accuracy_score(y_true_disease, y_pred_disease)
print(f"\nOverall Accuracy: {acc*100:.2f}%  ({sum(1 for a,b in zip(y_true_disease,y_pred_disease) if a==b)}/{N_EVAL})")

# ---- Macro / Weighted / Micro precision, recall, F1 ----
labels = sorted(set(y_true_disease) | set(y_pred_disease))

p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
    y_true_disease, y_pred_disease, labels=labels, average="macro", zero_division=0
)
p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(
    y_true_disease, y_pred_disease, labels=labels, average="weighted", zero_division=0
)
p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
    y_true_disease, y_pred_disease, labels=labels, average="micro", zero_division=0
)

print(f"\n{'Metric':<20} {'Macro':>10} {'Weighted':>10} {'Micro':>10}")
print("-" * 52)
print(f"{'Precision':<20} {p_macro*100:>9.2f}% {p_weighted*100:>9.2f}% {p_micro*100:>9.2f}%")
print(f"{'Recall':<20} {r_macro*100:>9.2f}% {r_weighted*100:>9.2f}% {r_micro*100:>9.2f}%")
print(f"{'F1':<20} {f_macro*100:>9.2f}% {f_weighted*100:>9.2f}% {f_micro*100:>9.2f}%")

# ---- Per-class report ----
print("\n" + "=" * 70)
print("PER-CLASS BREAKDOWN")
print("=" * 70)
report = classification_report(
    y_true_disease, y_pred_disease, labels=labels, zero_division=0, digits=3
)
print(report)

# ---- Confusion matrix ----
cm = confusion_matrix(y_true_disease, y_pred_disease, labels=labels)
print("\n" + "=" * 70)
print("CONFUSION MATRIX (rows=true, cols=pred)")
print("=" * 70)
header = "true \\ pred".ljust(35) + " ".join(f"{i:>4}" for i in range(len(labels)))
print(header)
for i, row in enumerate(cm):
    row_str = " ".join(f"{v:>4}" for v in row)
    print(f"{labels[i][:33]:<35} {row_str}")

# ---- Top confusions ----
print("\n" + "=" * 70)
print("TOP 15 MISCLASSIFICATIONS (true → predicted)")
print("=" * 70)
confusions = Counter()
for t, p in zip(y_true_disease, y_pred_disease):
    if t != p:
        confusions[(t, p)] += 1

for (t, p), cnt in confusions.most_common(15):
    print(f"  {cnt:>3}×  {t:<35} → {p}")

# ---- Per-class accuracy ----
print("\n" + "=" * 70)
print("PER-CLASS ACCURACY (sorted by accuracy)")
print("=" * 70)
per_class = {}
for cls in labels:
    idxs = [i for i, t in enumerate(y_true_disease) if t == cls]
    if not idxs:
        continue
    correct = sum(1 for i in idxs if y_pred_disease[i] == cls)
    per_class[cls] = (correct, len(idxs), correct/len(idxs))

for cls, (c, n, a) in sorted(per_class.items(), key=lambda x: x[1][2]):
    bar = "█" * int(a * 20)
    print(f"  {cls:<35} {c:>3}/{n:<3}  {a*100:>6.1f}%  {bar}")

# ---- Plant-level metrics (parent category) ----
print("\n" + "=" * 70)
print("PLANT-LEVEL METRICS (higher-level correctness)")
print("=" * 70)
plant_acc = accuracy_score(y_true_plant, y_pred_plant)
print(f"Plant name accuracy: {plant_acc*100:.2f}%")
plant_labels = sorted(set(y_true_plant) | set(y_pred_plant))
p_p, r_p, f_p, _ = precision_recall_fscore_support(
    y_true_plant, y_pred_plant, labels=plant_labels, average="macro", zero_division=0
)
print(f"Plant macro-F1:     {f_p*100:.2f}%")

# ---- Save metrics to disk ----
metrics = {
    "n_evaluated": N_EVAL,
    "disease_accuracy": float(acc),
    "disease_precision_macro": float(p_macro),
    "disease_recall_macro": float(r_macro),
    "disease_f1_macro": float(f_macro),
    "disease_precision_weighted": float(p_weighted),
    "disease_recall_weighted": float(r_weighted),
    "disease_f1_weighted": float(f_weighted),
    "plant_accuracy": float(plant_acc),
    "plant_f1_macro": float(f_p),
    "per_class_accuracy": {k: float(v[2]) for k, v in per_class.items()},
    "confusion_matrix": cm.tolist(),
    "labels": labels,
}

out_path = "/kaggle/working/evaluation_metrics.json"
with open(out_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\n Metrics saved to {out_path}")

# ---- Save examples for qualitative review ----
examples_path = "/kaggle/working/evaluation_examples.json"
with open(examples_path, "w") as f:
    json.dump(full_examples, f, indent=2)

print(f" All {N_EVAL} predictions saved to {examples_path}")

Evaluating 200 validation samples...
This takes ~4-5 seconds per sample on T4, ~15 minutes total.



Evaluating: 100%|██████████| 200/200 [13:25<00:00,  4.03s/it]


DISEASE-LEVEL METRICS

Overall Accuracy: 91.50%  (183/200)

Metric                    Macro   Weighted      Micro
----------------------------------------------------
Precision                86.61%     92.24%     91.50%
Recall                   87.06%     91.50%     91.50%
F1                       85.90%     91.43%     91.50%

PER-CLASS BREAKDOWN
                                      precision    recall  f1-score   support

                          apple scab      0.750     0.750     0.750         4
                      bacterial spot      0.955     1.000     0.977        21
                           black rot      0.923     0.923     0.923        13
                    cedar apple rust      1.000     1.000     1.000         2
 cercospora leaf spot gray leaf spot      0.857     1.000     0.923         6
                         common rust      1.000     1.000     1.000         6
                        early blight      0.714     0.500     0.588        10
                esca (bl

In [50]:
# CELL 8 (IMPROVED): Evaluate on validation set
from PIL import Image
import torch
from tqdm import tqdm

def get_disease(text):
    for line in text.split("\n"):
        if "Disease:" in line:
            return line.replace("**Disease:**", "").strip().lower()
    return "unknown"

model.eval()
correct = 0
total = 0

for i in tqdm(range(min(150, len(val_data))), desc="Evaluating"):
    sample = val_data[i]
    img = sample["image"]
    gt = sample["messages"][1]["content"][0]["text"]

    msgs = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Identify the plant disease in this image."}
    ]}]

    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[img], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, do_sample=False)

    pred = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    if get_disease(pred) == get_disease(gt):
        correct += 1
    total += 1

print(f"\nValidation accuracy (disease name): {correct}/{total} = {correct/total*100:.1f}%")

Evaluating: 100%|██████████| 150/150 [09:52<00:00,  3.95s/it]


Validation accuracy (disease name): 138/150 = 92.0%


In [51]:
# EVALUATION: Check real accuracy on validation set
from PIL import Image
import torch
from tqdm import tqdm

model.eval()
correct = 0
total = 0

for i in tqdm(range(min(100, len(val_data))), desc="Evaluating"):
    sample = val_data[i]
    img = sample["image"]
    gt = sample["messages"][1]["content"][0]["text"]

    msgs = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Identify the plant disease in this image."}
    ]}]

    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[img], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, do_sample=False)

    pred = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Extract disease line from both
    def get_disease(text):
        for line in text.split("\n"):
            if "Disease:" in line:
                return line.replace("**Disease:**", "").strip().lower()
        return "unknown"

    if get_disease(pred) == get_disease(gt):
        correct += 1
    total += 1

print(f"\nValidation accuracy (disease name): {correct}/{total} = {correct/total*100:.1f}%")

Evaluating: 100%|██████████| 100/100 [06:38<00:00,  3.98s/it]


Validation accuracy (disease name): 93/100 = 93.0%


In [52]:
# CELL 9 (IMPROVED): Merge adapter v2 into base model
import torch, os, json

# ---- torchao bypass ----
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao as _lt
    _lt.is_torchao_available = lambda: False
except Exception:
    pass
try:
    import peft.tuners.lora.model as _lm
    if hasattr(_lm, "is_torchao_available"):
        _lm.is_torchao_available = lambda: False
except Exception:
    pass

from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
adapter_dir = "/kaggle/working/smolvlm-plantvillage-lora-v2"   # ← updated
merged_dir = "/kaggle/working/smolvlm-plantvillage-merged-v2"

print("Loading base model in fp16...")
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
)

print("Merging LoRA v2...")
merged_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(merged_dir)

# ---- Copy processor + tokenizer files from ORIGINAL base model ----
# This is CRITICAL for GGUF conversion to find the tokenizer
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.save_pretrained(merged_dir)

print(f"Merged model saved to {merged_dir}")
print(f"Contents: {sorted(os.listdir(merged_dir))}")

st_path = os.path.join(merged_dir, "model.safetensors")
if os.path.exists(st_path):
    print(f"model.safetensors: {os.path.getsize(st_path)/1e9:.2f} GB")

Loading base model in fp16...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Merging LoRA v2...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to /kaggle/working/smolvlm-plantvillage-merged-v2
Contents: ['chat_template.jinja', 'config.json', 'generation_config.json', 'model.safetensors', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json']
model.safetensors: 1.02 GB


In [53]:
# CELL 10: Verify or build llama.cpp in /kaggle/working
import os

os.chdir("/kaggle/working")
llama_dir = "/kaggle/working/llama.cpp"
quantize_bin = "/kaggle/working/llama.cpp/build/bin/llama-quantize"
convert_script = "/kaggle/working/llama.cpp/convert_hf_to_gguf.py"

if os.path.exists(llama_dir):
    print(f"✓ llama.cpp exists at {llama_dir}")
    print(f"  convert script: {os.path.exists(convert_script)}")
    print(f"  quantize binary: {os.path.exists(quantize_bin)}")
else:
    print("✗ llama.cpp not found — cloning and building now...")
    !git clone https://github.com/ggerganov/llama.cpp
    !cd llama.cpp && pip install -q -r requirements.txt
    !cd llama.cpp && cmake -B build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF
    !cd llama.cpp && cmake --build build --config Release -j4
    print("✓ Build complete")

# Sanity check
for p in [convert_script, quantize_bin]:
    if os.path.exists(p):
        print(f"  ✅ {p}")
    else:
        print(f"  ❌ MISSING: {p}")

✓ llama.cpp exists at /kaggle/working/llama.cpp
  convert script: True
  quantize binary: True
  ✅ /kaggle/working/llama.cpp/convert_hf_to_gguf.py
  ✅ /kaggle/working/llama.cpp/build/bin/llama-quantize


In [54]:
# CELL 10: Clone llama.cpp and build tools
import os

if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp

!cd llama.cpp && pip install -q -r requirements.txt
!cd llama.cpp && cmake -B build -DLLAMA_CURL=OFF
!cd llama.cpp && cmake --build build --config Release -j4
print("llama.cpp built.")

-- llama.cpp version: 0.6.0-dev
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.26.0
-- ggml commit:  03aa006ac
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: llama-app
-- Configuring done (0.5s)
-- Generating done (0.4s)
-- Build files have been written to: /kaggle/working/llama.cpp/build
[  1%] Built target llama-common-base
[  3%] Built target vendor-hash
[  3%] Built target cpp-httplib
[  4%] Provisioning UI assets
[  6%] Built target ggml-base
-- UI: HF stamp matches 'ggml-org/llama-ui|b11503', skipping HF fetch
[  6%] Built target llama-llava-cli
[  6%] Built target llama-gemma3-cli
[  7%] Built target llama-qwen2vl-cli
[  7%] Built target llama-minicpmv-cli
[ 11%] Built target ggml-cpu

In [59]:
# CELL 11-FIX-TOKENIZER: Patch extra_special_tokens from list to dict
import json, os

merged_dir = "/kaggle/working/smolvlm-plantvillage-merged-v2"
tokenizer_config_path = os.path.join(merged_dir, "tokenizer_config.json")

with open(tokenizer_config_path) as f:
    config = json.load(f)

print("Before patch:")
print(f"  extra_special_tokens: {config.get('extra_special_tokens', 'NOT PRESENT')}")
print(f"  type: {type(config.get('extra_special_tokens'))}")

# Fix: Convert list to dict, or remove entirely
if isinstance(config.get("extra_special_tokens"), list):
    # Option A: Convert to dict with token names as keys
    tokens_list = config["extra_special_tokens"]
    config["extra_special_tokens"] = {t: t for t in tokens_list}
    print(f"\nConverted list to dict: {config['extra_special_tokens']}")
elif "extra_special_tokens" not in config:
    print("\nNo extra_special_tokens field — nothing to fix.")
else:
    print(f"\nField is already a {type(config.get('extra_special_tokens'))} — no fix needed.")

# Save back
with open(tokenizer_config_path, "w") as f:
    json.dump(config, f, indent=2)

print("\n tokenizer_config.json patched.")

# Also check if there's a tokenizer_config.json in the adapter directory (if using adapter)
# and patch that too if it exists
adapter_config = os.path.join("/kaggle/working/smolvlm-plantvillage-lora-v2", "tokenizer_config.json")
if os.path.exists(adapter_config):
    with open(adapter_config) as f:
        ac = json.load(f)
    if isinstance(ac.get("extra_special_tokens"), list):
        ac["extra_special_tokens"] = {t: t for t in ac["extra_special_tokens"]}
        with open(adapter_config, "w") as f:
            json.dump(ac, f, indent=2)
        print(" Also patched adapter tokenizer_config.json")

Before patch:
  extra_special_tokens: ['<fake_token_around_image>', '<image>', '<end_of_utterance>']
  type: <class 'list'>

Converted list to dict: {'<fake_token_around_image>': '<fake_token_around_image>', '<image>': '<image>', '<end_of_utterance>': '<end_of_utterance>'}

 tokenizer_config.json patched.
 Also patched adapter tokenizer_config.json


In [60]:
# CELL 11-RETRY: Convert after patching tokenizer_config.json
import subprocess, os, sys

merged_dir = "/kaggle/working/smolvlm-plantvillage-merged-v2"
out_f16 = "/kaggle/working/smolvlm-plantvillage-f16.gguf"
llama_cpp_dir = "/kaggle/working/llama.cpp"
lib_dir = "/kaggle/working/convert_libs"

if os.path.exists(out_f16):
    os.remove(out_f16)

env = os.environ.copy()
env["PYTHONPATH"] = lib_dir + ":" + env.get("PYTHONPATH", "")

print("=== Converting with patched tokenizer_config.json ===")
result = subprocess.run(
    [sys.executable, "convert_hf_to_gguf.py", merged_dir,
     "--outfile", out_f16, "--outtype", "f16"],
    cwd=llama_cpp_dir,
    capture_output=True, text=True,
    env=env,
)

print("Return code:", result.returncode)
if result.returncode != 0:
    print("\nSTDERR (last 3000 chars):")
    print(result.stderr[-3000:])
    print("\nSTDOUT (last 1000 chars):")
    print(result.stdout[-1000:])
else:
    print(f"\n {out_f16}  ({os.path.getsize(out_f16)/1e6:.1f} MB)")

=== Converting with patched tokenizer_config.json ===
Return code: 0

 /kaggle/working/smolvlm-plantvillage-f16.gguf  (820.4 MB)


In [58]:
# CELL 11-FIX: Convert with transformers 4.x via --target + PYTHONPATH
import subprocess, os, sys

merged_dir = "/kaggle/working/smolvlm-plantvillage-merged-v2"
out_f16 = "/kaggle/working/smolvlm-plantvillage-f16.gguf"
llama_cpp_dir = "/kaggle/working/llama.cpp"
lib_dir = "/kaggle/working/convert_libs"

# Step 1: Install transformers 4.x into a custom directory
os.makedirs(lib_dir, exist_ok=True)

print("Installing transformers 4.x into custom dir...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--target", lib_dir,
    "transformers>=4.36.0,<5.0.0",
    "sentencepiece",
    "protobuf",
    "safetensors",
    "huggingface_hub",
    "gguf",
], check=True)

# Step 2: Clean up previous failed output
if os.path.exists(out_f16):
    os.remove(out_f16)

# Step 3: Set PYTHONPATH so subprocess picks up the custom transformers
env = os.environ.copy()
env["PYTHONPATH"] = lib_dir + ":" + env.get("PYTHONPATH", "")

print(f"\n=== Converting main model with transformers 4.x (via PYTHONPATH) ===")
result = subprocess.run(
    [sys.executable, "convert_hf_to_gguf.py", merged_dir,
     "--outfile", out_f16, "--outtype", "f16"],
    cwd=llama_cpp_dir,
    capture_output=True, text=True,
    env=env,
)

print("Return code:", result.returncode)
if result.returncode != 0:
    print("STDERR (last 4000 chars):")
    print(result.stderr[-4000:])
    print("\nSTDOUT (last 2000 chars):")
    print(result.stdout[-2000:])
else:
    print(f" {out_f16}  ({os.path.getsize(out_f16)/1e6:.1f} MB)")

Installing transformers 4.x into custom dir...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 32.5 MB/s eta 0:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 5.1.0 requires fsspec[http]<=2026.7.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
datasets 5.1.0 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.36.2 which is incompatible.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.
google-cloud-aiplatform 2.3.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
litellm 1.85.7 requires aiohttp<4.0,>=3.10, but you have aiohttp 3.9.5 which is incompatible.
litellm 1.85.7 requires openai<3.0.0,>=


=== Converting main model with transformers 4.x (via PYTHONPATH) ===
Return code: 1
STDERR (last 4000 chars):
king/llama.cpp/conversion/base.py", line 2093, in _set_vocab_sentencepiece
    tokens, scores, toktypes = self._create_vocab_sentencepiece()
                               ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/kaggle/working/llama.cpp/conversion/base.py", line 2110, in _create_vocab_sentencepiece
    raise FileNotFoundError(f"File not found: {tokenizer_path}")
FileNotFoundError: File not found: /kaggle/working/smolvlm-plantvillage-merged-v2/tokenizer.model

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/kaggle/working/llama.cpp/conversion/llama.py", line 139, in set_vocab
    self._set_vocab_llama_hf()
    ~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/kaggle/working/llama.cpp/conversion/base.py", line 2195, in _set_vocab_llama_hf
    vocab = gguf.LlamaHfVocab(self.dir_model)
  File "/kaggle/working/llama.cpp/gguf-py

In [61]:
# CELL 12: Quantize main model to Q4_K_M and mmproj to Q8_0
import os, subprocess

f16 = "/kaggle/working/smolvlm-plantvillage-f16.gguf"
mmproj_f16 = "/kaggle/working/mmproj-smolvlm-f16.gguf"
q4 = "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf"
mmproj_q8 = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"
quantize_bin = "/kaggle/working/llama.cpp/build/bin/llama-quantize"

if not os.path.exists(f16):
    raise FileNotFoundError(f"F16 main model missing: {f16}")
if not os.path.exists(quantize_bin):
    raise FileNotFoundError(f"llama-quantize missing: {quantize_bin}")

print("Quantizing main model to Q4_K_M...")
r = subprocess.run([quantize_bin, f16, q4, "Q4_K_M"], capture_output=True, text=True)
print("RC:", r.returncode)
if r.returncode != 0:
    print("STDERR:", r.stderr[-1500:])
elif os.path.exists(q4):
    print(f" {q4}  ({os.path.getsize(q4)/1e6:.1f} MB)")

if os.path.exists(mmproj_f16):
    print("\nQuantizing mmproj to Q8_0...")
    r = subprocess.run([quantize_bin, mmproj_f16, mmproj_q8, "Q8_0"], capture_output=True, text=True)
    print("RC:", r.returncode)
    if os.path.exists(mmproj_q8):
        print(f" {mmproj_q8}  ({os.path.getsize(mmproj_q8)/1e6:.1f} MB)")

print("\n=== Final GGUF files ===")
for f in [q4, mmproj_q8]:
    if os.path.exists(f):
        print(f"   {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.1f} MB)")
    else:
        print(f"   MISSING: {os.path.basename(f)}")

Quantizing main model to Q4_K_M...
RC: 0
 /kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf  (303.3 MB)

Quantizing mmproj to Q8_0...
RC: 0
 /kaggle/working/mmproj-smolvlm-Q8_0.gguf  (108.8 MB)

=== Final GGUF files ===
   smolvlm-plantvillage-Q4_K_M.gguf  (303.3 MB)
   mmproj-smolvlm-Q8_0.gguf  (108.8 MB)


In [62]:
# CELL 13: Package everything for Kaggle Output
import os, shutil, zipfile

out_dir = "/kaggle/working/android_export"
os.makedirs(out_dir, exist_ok=True)

required = [
    "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf",
    "/kaggle/working/mmproj-smolvlm-Q8_0.gguf",
]
optional_dirs = [
    "/kaggle/working/smolvlm-plantvillage-lora-v2",
]

missing = [f for f in required if not os.path.exists(f)]
if missing:
    print(f"⚠ Missing required files: {missing}")
else:
    for f in required:
        shutil.copy(f, out_dir)
    for d in optional_dirs:
        if os.path.exists(d):
            shutil.copytree(d, os.path.join(out_dir, os.path.basename(d)), dirs_exist_ok=True)

    zip_path = "/kaggle/working/smolvlm-plantvillage-android.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(out_dir):
            for f in files:
                full = os.path.join(root, f)
                rel = os.path.relpath(full, "/kaggle/working")
                z.write(full, rel)

    print("=" * 60)
    print("PACKAGED FOR KAGGLE OUTPUT")
    print("=" * 60)
    print(f"\nandroid_export/ contents:")
    for root, dirs, files in os.walk(out_dir):
        for f in sorted(files):
            p = os.path.join(root, f)
            rel = os.path.relpath(p, out_dir)
            print(f"  {rel}  ({os.path.getsize(p)/1e6:.1f} MB)")

    print(f"\n Zip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
    print("\n HOW TO DOWNLOAD FROM KAGGLE:")
    print("  1. Click 'Save Version' (top right)")
    print("  2. Choose 'Save & Run All (Commit)'")
    print("  3. Wait for completion")
    print("  4. Open the notebook's 'Output' tab")
    print("  5. Download 'smolvlm-plantvillage-android.zip'")

PACKAGED FOR KAGGLE OUTPUT

android_export/ contents:
  mmproj-smolvlm-Q8_0.gguf  (108.8 MB)
  smolvlm-plantvillage-Q4_K_M.gguf  (303.3 MB)
  smolvlm-plantvillage-lora-v2/README.md  (0.0 MB)
  smolvlm-plantvillage-lora-v2/adapter_config.json  (0.0 MB)
  smolvlm-plantvillage-lora-v2/adapter_model.safetensors  (19.2 MB)
  smolvlm-plantvillage-lora-v2/chat_template.jinja  (0.0 MB)
  smolvlm-plantvillage-lora-v2/processor_config.json  (0.0 MB)
  smolvlm-plantvillage-lora-v2/tokenizer.json  (3.5 MB)
  smolvlm-plantvillage-lora-v2/tokenizer_config.json  (0.0 MB)

 Zip: /kaggle/working/smolvlm-plantvillage-android.zip  (416.6 MB)

 HOW TO DOWNLOAD FROM KAGGLE:
  1. Click 'Save Version' (top right)
  2. Choose 'Save & Run All (Commit)'
  3. Wait for completion
  4. Open the notebook's 'Output' tab
  5. Download 'smolvlm-plantvillage-android.zip'


In [64]:
# CELL 14A: Verify exported GGUF files via llama-mtmd-cli
import subprocess
import os

main_model = "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf"
mmproj_model = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"
test_image = "/kaggle/working/test_leaf.jpg"
llama_cpp_dir = "/kaggle/working/llama.cpp"
mtmd_cli = os.path.join(llama_cpp_dir, "build", "bin", "llama-mtmd-cli")

# Verify binaries and models exist
for path, label in [
    (main_model, "Main model Q4_K_M"),
    (mmproj_model, "Vision projector Q8_0"),
    (mtmd_cli, "llama-mtmd-cli binary"),
]:
    if os.path.exists(path):
        print(f"OK      {label}: {os.path.getsize(path)/1e6:.1f} MB")
    else:
        print(f"MISSING {label}: {path}")
        raise FileNotFoundError(path)

# Export a test image from the validation set
if not os.path.exists(test_image):
    test_sample = val_data[0]
    test_sample["image"].save(test_image, "JPEG")
    print(f"\nExported test image: {test_image}")
    print(f"Ground truth: {test_sample['messages'][1]['content'][0]['text']}")

# Run inference
cmd = [
    mtmd_cli,
    "-m", main_model,
    "--mmproj", mmproj_model,
    "--image", test_image,
    "-p", "Identify the plant disease in this image.",
    "-n", "150",
    "--temp", "0.0",
]

print("\n" + "=" * 60)
print("RUNNING INFERENCE")
print("=" * 60)

result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)

if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
else:
    print("MODEL OUTPUT:")
    print(result.stdout[-2000:])

print("\n" + "=" * 60)
print("VERIFICATION COMPLETE")
print("=" * 60)

OK      Main model Q4_K_M: 303.3 MB
OK      Vision projector Q8_0: 108.8 MB
OK      llama-mtmd-cli binary: 0.1 MB

Exported test image: /kaggle/working/test_leaf.jpg
Ground truth: **Plant:** Tomato
**Disease:** Tomato Yellow Leaf Curl Virus
**Symptoms:** Visible symptoms consistent with Tomato Yellow Leaf Curl Virus.

RUNNING INFERENCE
MODEL OUTPUT:

 Tomato.




VERIFICATION COMPLETE


In [65]:
# CELL 14A-FIX: Inference with explicit SmolVLM chat template
import subprocess, os

main_model = "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf"
mmproj_model = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"
test_image = "/kaggle/working/test_leaf.jpg"
llama_cpp_dir = "/kaggle/working/llama.cpp"
mtmd_cli = os.path.join(llama_cpp_dir, "build", "bin", "llama-mtmd-cli")

# SmolVLM chat template — matches the format used in training
# The token <image> marks where the vision tokens should be inserted
prompt = "<|im_start|>User:<image>Identify the plant disease in this image.<end_of_utterance>\nAssistant:"

# Get the chat template from the model itself
# llama-mtmd-cli has a -p flag for raw prompt, but we need chat mode
cmd = [
    mtmd_cli,
    "-m", main_model,
    "--mmproj", mmproj_model,
    "--image", test_image,
    "-p", prompt,
    "-n", "200",
    "--temp", "0.0",
    "--top-k", "1",
    "--repeat-penalty", "1.0",
]

print("Running inference with explicit SmolVLM chat template...")
print(f"Prompt: {repr(prompt)}")
print()

result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)

print("=" * 60)
print("MODEL OUTPUT")
print("=" * 60)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
else:
    # llama-mtmd-cli prints generation info to stderr, actual text to stdout
    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("\nSTDERR (last 1000 chars):")
        print(result.stderr[-1000:])

Running inference with explicit SmolVLM chat template...
Prompt: '<|im_start|>User:<image>Identify the plant disease in this image.<end_of_utterance>\nAssistant:'

MODEL OUTPUT
STDOUT:

 **Plant:** Tomato
**Disease:** Bacterial spot
**Symptoms:** Visible symptoms consistent with Bacterial spot.




STDERR (last 1000 chars):
2852 ms
0.31.993.469 I encoding mtmd batch, n_chunks = 1 (done = 19, total = 35)
0.34.752.301 I mtmd batch encoding done in 2759 ms
0.35.448.616 I encoding mtmd batch, n_chunks = 1 (done = 21, total = 35)
0.38.204.841 I mtmd batch encoding done in 2756 ms
0.38.904.455 I encoding mtmd batch, n_chunks = 1 (done = 23, total = 35)
0.41.688.694 I mtmd batch encoding done in 2785 ms
0.42.434.925 I encoding mtmd batch, n_chunks = 1 (done = 25, total = 35)
0.45.160.668 I mtmd batch encoding done in 2726 ms
0.45.889.246 I encoding mtmd batch, n_chunks = 1 (done = 27, total = 35)
0.48.615.327 I mtmd batch encoding done in 2726 ms
0.49.401.859 I encoding mtmd batch, n_chunks =

In [66]:
# CELL 14B: Print SmolVLM chat template and test tokenization
from transformers import AutoProcessor
import json

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-500M-Instruct")

# Print the chat template
if hasattr(processor.tokenizer, "chat_template"):
    print("Chat template:")
    print(processor.tokenizer.chat_template)
else:
    print("No chat template found on tokenizer.")

# Show the full chat template from chat_template.jinja
import os
jt_path = "/kaggle/working/smolvlm-plantvillage-merged-v2/chat_template.jinja"
if os.path.exists(jt_path):
    print("\nchat_template.jinja file:")
    with open(jt_path) as f:
        print(f.read())

Chat template:
<|im_start|>{% for message in messages %}{{message['role'] | capitalize}}{% if message['content'][0]['type'] == 'image' %}{{':'}}{% else %}{{': '}}{% endif %}{% for line in message['content'] %}{% if line['type'] == 'text' %}{{line['text']}}{% elif line['type'] == 'image' %}{{ '<image>' }}{% endif %}{% endfor %}<end_of_utterance>
{% endfor %}{% if add_generation_prompt %}{{ 'Assistant:' }}{% endif %}

chat_template.jinja file:
<|im_start|>{% for message in messages %}{{message['role'] | capitalize}}{% if message['content'][0]['type'] == 'image' %}{{':'}}{% else %}{{': '}}{% endif %}{% for line in message['content'] %}{% if line['type'] == 'text' %}{{line['text']}}{% elif line['type'] == 'image' %}{{ '<image>' }}{% endif %}{% endfor %}<end_of_utterance>
{% endfor %}{% if add_generation_prompt %}{{ 'Assistant:' }}{% endif %}


In [67]:
# CELL 14C: Verbose inference to diagnose truncation
import subprocess, os

main_model = "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf"
mmproj_model = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"
test_image = "/kaggle/working/test_leaf.jpg"
mtmd_cli = "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli"

cmd = [
    mtmd_cli,
    "-m", main_model,
    "--mmproj", mmproj_model,
    "--image", test_image,
    "-p", "Identify the plant disease in this image.",
    "-n", "200",
    "--temp", "0.0",
    "--verbose",
]

result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)

print("Return code:", result.returncode)
print("\n--- STDOUT ---")
print(result.stdout)
print("\n--- STDERR (last 2500 chars) ---")
print(result.stderr[-2500:])

Return code: 0

--- STDOUT ---

 Tomato.




--- STDERR (last 2500 chars) ---
12)
0.48.871.254 D clip_encode: output embedding shape [960, 64, 1]
0.48.871.310 I mtmd batch encoding done in 2756 ms
0.48.871.319 I decoding image batch 1/1, n_tokens_batch = 64
0.49.552.899 I image decoded (batch 1/1) in 682 ms
0.49.612.247 D media chunk 29 not found in existing batch, creating new batch
0.49.612.265 I encoding mtmd batch, n_chunks = 1 (done = 29, total = 35)
0.49.612.990 D mtmd_batch_encode_impl: encoding batch with 1 entries and total 64 tokens
0.49.613.854 D clip_encode: copying image 1/1 to input buffer (nx=512, ny=512)
0.52.445.328 D clip_encode: output embedding shape [960, 64, 1]
0.52.445.384 I mtmd batch encoding done in 2833 ms
0.52.445.394 I decoding image batch 1/1, n_tokens_batch = 64
0.53.152.031 I image decoded (batch 1/1) in 707 ms
0.53.215.308 D media chunk 31 not found in existing batch, creating new batch
0.53.215.321 I encoding mtmd batch, n_chunks = 1 (done = 31, total 

In [68]:
# CELL 15: Verify GGUF inference accuracy on 20 validation samples
import subprocess, os
from collections import Counter

main_model = "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf"
mmproj_model = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"
mtmd_cli = "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli"

# Chat template that matches SmolVLM training format
def build_prompt():
    return "<|im_start|>User:<image>Identify the plant disease in this image.<end_of_utterance>\nAssistant:"

def get_disease(text):
    for line in text.split("\n"):
        if "Disease:" in line:
            return line.replace("**Disease:**", "").replace("*", "").strip().lower()
    return "unknown"

# Run 20 samples
N = 20
correct = 0
total = 0
results = []

print(f"Running inference on {N} validation samples...")
print("This takes about 2-4 seconds per sample on CPU.\n")

for i in range(N):
    sample = val_data[i]
    img_path = f"/kaggle/working/gguf_test_{i}.jpg"
    sample["image"].save(img_path, "JPEG")

    gt_text = sample["messages"][1]["content"][0]["text"]
    gt_disease = get_disease(gt_text)

    cmd = [
        mtmd_cli,
        "-m", main_model,
        "--mmproj", mmproj_model,
        "--image", img_path,
        "-p", build_prompt(),
        "-n", "100",
        "--temp", "0.0",
        "--top-k", "1",
    ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        pred_text = result.stdout.strip()
        pred_disease = get_disease(pred_text)

        match = pred_disease == gt_disease
        if match:
            correct += 1
        total += 1

        results.append({
            "idx": i,
            "gt": gt_disease,
            "pred": pred_disease,
            "match": match,
        })

        status = "OK " if match else "MISS"
        print(f"{status} [{i:2d}] gt={gt_disease:<35} pred={pred_disease}")

    except subprocess.TimeoutExpired:
        print(f"TIMEOUT [{i}]")
        total += 1

print("\n" + "=" * 60)
print(f"GGUF Inference Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
print("=" * 60)

# Show mismatches
print("\nMismatches:")
for r in results:
    if not r["match"]:
        print(f"  [{r['idx']:2d}] {r['gt']:<35} -> {r['pred']}")

Running inference on 20 validation samples...
This takes about 2-4 seconds per sample on CPU.

MISS [ 0] gt=tomato yellow leaf curl virus       pred=bacterial spot
MISS [ 1] gt=healthy                             pred=powdery mildew
MISS [ 2] gt=septoria leaf spot                  pred=bacterial spot
MISS [ 3] gt=healthy                             pred=powdery mildew
MISS [ 4] gt=healthy                             pred=powdery mildew
MISS [ 5] gt=black rot                           pred=powdery mildew
MISS [ 6] gt=leaf mold                           pred=powdery mildew
MISS [ 7] gt=late blight                         pred=powdery mildew
MISS [ 8] gt=healthy                             pred=apple scab
MISS [ 9] gt=bacterial spot                      pred=powdery mildew
MISS [10] gt=healthy                             pred=apple scab
MISS [11] gt=early blight                        pred=bacterial spot
MISS [12] gt=healthy                             pred=powdery mildew
MISS [13] gt=tom

In [69]:
# CELL 16: Compare GGUF inference vs PyTorch inference on 10 samples
import torch
from PIL import Image

def get_disease(text):
    for line in text.split("\n"):
        if "Disease:" in line:
            return line.replace("**Disease:**", "").replace("*", "").strip().lower()
    return "unknown"

N = 10
gguf_correct = 0
torch_correct = 0
agreements = 0

print(f"Comparing GGUF vs PyTorch on {N} samples...\n")

for i in range(N):
    sample = val_data[i]
    img = sample["image"]
    img_path = f"/kaggle/working/compare_{i}.jpg"
    img.save(img_path, "JPEG")

    gt_text = sample["messages"][1]["content"][0]["text"]
    gt_disease = get_disease(gt_text)

    # --- GGUF inference ---
    cmd = [
        "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli",
        "-m", "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf",
        "--mmproj", "/kaggle/working/mmproj-smolvlm-Q8_0.gguf",
        "--image", img_path,
        "-p", "<|im_start|>User:<image>Identify the plant disease in this image.<end_of_utterance>\nAssistant:",
        "-n", "100", "--temp", "0.0", "--top-k", "1",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    gguf_disease = get_disease(result.stdout)

    # --- PyTorch inference ---
    msgs = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Identify the plant disease in this image."}
    ]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[img], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    torch_text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    torch_disease = get_disease(torch_text)

    gguf_match = gguf_disease == gt_disease
    torch_match = torch_disease == gt_disease
    agree = gguf_disease == torch_disease

    gguf_correct += int(gguf_match)
    torch_correct += int(torch_match)
    agreements += int(agree)

    print(f"[{i:2d}] gt={gt_disease:<30}")
    print(f"       GGUF  : {gguf_disease:<30} {'OK' if gguf_match else 'MISS'}")
    print(f"       PyTorch: {torch_disease:<30} {'OK' if torch_match else 'MISS'}")

print("\n" + "=" * 60)
print(f"GGUF  accuracy: {gguf_correct}/{N} = {gguf_correct/N*100:.1f}%")
print(f"Torch accuracy: {torch_correct}/{N} = {torch_correct/N*100:.1f}%")
print(f"Agreement:      {agreements}/{N} = {agreements/N*100:.1f}%")
print("=" * 60)

Comparing GGUF vs PyTorch on 10 samples...

[ 0] gt=tomato yellow leaf curl virus 
       GGUF  : bacterial spot                 MISS
       PyTorch: unknown                        MISS
[ 1] gt=healthy                       
       GGUF  : powdery mildew                 MISS
       PyTorch: healthy                        OK
[ 2] gt=septoria leaf spot            
       GGUF  : bacterial spot                 MISS
       PyTorch: septoria leaf spot             OK
[ 3] gt=healthy                       
       GGUF  : powdery mildew                 MISS
       PyTorch: healthy                        OK
[ 4] gt=healthy                       
       GGUF  : powdery mildew                 MISS
       PyTorch: healthy                        OK
[ 5] gt=black rot                     
       GGUF  : powdery mildew                 MISS
       PyTorch: black rot                      OK
[ 6] gt=leaf mold                     
       GGUF  : powdery mildew                 MISS
       PyTorch: unknown 

In [70]:
# CELL 17: Reference inference function (mirror for Android integration)
def infer_plant_disease(image_path: str, main_model: str, mmproj: str) -> dict:
    """
    Reference inference function that mirrors what Android JNI will do.

    Android equivalent uses:
      - LlamaContext.loadModel(mainModelPath, mmprojPath)
      - ctx.processImage(bitmap)
      - ctx.generate(prompt, maxTokens=100, temperature=0.0)
    """
    prompt = "<|im_start|>User:<image>Identify the plant disease in this image.<end_of_utterance>\nAssistant:"

    cmd = [
        "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli",
        "-m", main_model,
        "--mmproj", mmproj,
        "--image", image_path,
        "-p", prompt,
        "-n", "100",
        "--temp", "0.0",
        "--top-k", "1",
    ]

    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    text = result.stdout.strip()

    # Parse structured output
    parsed = {"plant": "", "disease": "", "symptoms": "", "raw": text}
    for line in text.split("\n"):
        if "Plant:" in line:
            parsed["plant"] = line.split("Plant:")[-1].replace("*", "").strip()
        elif "Disease:" in line:
            parsed["disease"] = line.split("Disease:")[-1].replace("*", "").strip()
        elif "Symptoms:" in line:
            parsed["symptoms"] = line.split("Symptoms:")[-1].replace("*", "").strip()

    return parsed

# Test the function
test_result = infer_plant_disease(
    "/kaggle/working/test_leaf.jpg",
    "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf",
    "/kaggle/working/mmproj-smolvlm-Q8_0.gguf",
)

print("Parsed output:")
for key, value in test_result.items():
    if key != "raw":
        print(f"  {key}: {value}")
print(f"\nRaw text:\n{test_result['raw']}")

Parsed output:
  plant: Tomato
  disease: Bacterial spot
  symptoms: Visible symptoms consistent with Bacterial spot.

Raw text:
**Plant:** Tomato
**Disease:** Bacterial spot
**Symptoms:** Visible symptoms consistent with Bacterial spot.


In [71]:
# CELL 18: Inspect mmproj GGUF metadata for configuration mismatches
import struct, os

mmproj_path = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"

# GGUF files can be inspected with llama.cpp's gguf tool
# Use the Python gguf library
import subprocess, sys
sys.path.insert(0, "/kaggle/working/llama.cpp/gguf-py")

from gguf import GGUFReader

reader = GGUFReader(mmproj_path)

print("=" * 70)
print("MMPROJ METADATA")
print("=" * 70)
for field in reader.fields.values():
    if field.name.startswith("clip.") or field.name.startswith("general."):
        try:
            value = field.contents()
        except Exception:
            value = "<unreadable>"
        print(f"  {field.name:50s} = {value}")

print("\n" + "=" * 70)
print("TENSOR SHAPES (first 10)")
print("=" * 70)
for i, tensor in enumerate(reader.tensors[:10]):
    print(f"  {tensor.name:50s} shape={list(tensor.shape)} type={tensor.tensor_type}")

print(f"\nTotal tensors: {len(reader.tensors)}")

MMPROJ METADATA
  general.architecture                               = clip
  general.type                                       = mmproj
  general.name                                       = Smolvlm Plantvillage Merged v2
  general.version                                    = v2
  general.basename                                   = smolvlm-plantvillage-merged
  general.size_label                                 = 98M
  clip.has_vision_encoder                            = True
  clip.vision.projection_dim                         = 960
  clip.vision.image_size                             = 512
  clip.vision.patch_size                             = 16
  clip.vision.embedding_length                       = 768
  clip.vision.feed_forward_length                    = 3072
  clip.vision.block_count                            = 12
  clip.vision.attention.head_count                   = 12
  clip.vision.image_mean                             = [0.5, 0.5, 0.5]
  clip.vision.image_std           

In [72]:
# CELL 19: Verbose inference to check image token insertion
import subprocess, os

main_model = "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf"
mmproj_model = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"
test_image = "/kaggle/working/test_leaf_16.jpg"  # this sample was classified correctly
mtmd_cli = "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli"

# Ensure the test image exists
val_data[16]["image"].save(test_image, "JPEG")

prompt = "<|im_start|>User:<image>Identify the plant disease in this image.<end_of_utterance>\nAssistant:"

cmd = [
    mtmd_cli,
    "-m", main_model,
    "--mmproj", mmproj_model,
    "--image", test_image,
    "-p", prompt,
    "-n", "100",
    "--temp", "0.0",
    "--verbose",
]

result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)

# Extract key lines from stderr
print("=" * 70)
print("KEY STDOUT")
print("=" * 70)
print(result.stdout)

print("=" * 70)
print("TOKENIZATION AND ENCODING LOG")
print("=" * 70)

for line in result.stderr.split("\n"):
    # Show only lines about image tokens, n_tokens, context
    keywords = ["n_tokens", "image", "clip", "mtmd", "n_past", "batch"]
    if any(kw in line.lower() for kw in keywords):
        # Skip progress bars
        if "%" in line and "encoding" in line.lower():
            continue
        print(line)

KEY STDOUT

 **Plant:** Peach
**Disease:** Bacterial spot
**Symptoms:** Visible symptoms consistent with Bacterial spot.



TOKENIZATION AND ENCODING LOG
0.00.089.401 D load: control token:  49189 '<fake_token_around_image>' is not marked as EOG
0.00.089.879 D load: control token:  49190 '<image>' is not marked as EOG
0.00.137.300 I llama_context: n_batch               = 2048
0.00.137.301 I llama_context: n_ubatch              = 512
0.00.138.486 D sched_reserve: worst-case: n_tokens = 512, n_seqs = 1, n_outputs = 1
0.00.138.489 D graph_reserve: reserving a graph for ubatch with n_tokens =    1, n_seqs =  1, n_outputs =    1
0.00.139.834 D graph_reserve: reserving a graph for ubatch with n_tokens =    1, n_seqs =  1, n_outputs =    1
0.00.141.287 D graph_reserve: reserving a graph for ubatch with n_tokens =    1, n_seqs =  1, n_outputs =    1
0.00.142.647 D graph_reserve: reserving a graph for ubatch with n_tokens =    1, n_seqs =  1, n_outputs =    1
0.00.143.889 D graph_reserve: reser

In [73]:
# CELL 23: Correct inference — pass raw user message, not ChatML
import subprocess, os

main_model = "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf"
mmproj_model = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"
test_image = "/kaggle/working/test_leaf_16.jpg"
mtmd_cli = "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli"

if not os.path.exists(test_image):
    val_data[16]["image"].save(test_image, "JPEG")

# CORRECT: pass only the raw user message. The CLI applies the chat template.
raw_message = "Identify the plant disease in this image."

cmd = [
    mtmd_cli,
    "-m", main_model,
    "--mmproj", mmproj_model,
    "--image", test_image,
    "-p", raw_message,
    "-n", "150",
    "--temp", "0.0",
    "--top-k", "1",
]

print("Running inference with raw user message...")
result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)

print("=" * 70)
print("MODEL OUTPUT")
print("=" * 70)
print(result.stdout)

print("=" * 70)
print("FORMATTED PROMPT (from stderr — verify single template application)")
print("=" * 70)
for line in result.stderr.split("\n"):
    if "formatted_chat.prompt" in line or "chat_add_and_format" in line:
        print(line)

print("=" * 70)
print("GROUND TRUTH")
print("=" * 70)
print(val_data[16]["messages"][1]["content"][0]["text"])

Running inference with raw user message...
MODEL OUTPUT

 Pepper.



FORMATTED PROMPT (from stderr — verify single template application)
GROUND TRUTH
**Plant:** Pepper,_bell
**Disease:** Bacterial spot
**Symptoms:** Visible symptoms consistent with Bacterial spot.


In [74]:
# CELL 23-FIX: Force SmolVLM chat template via file
import subprocess, os

main_model = "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf"
mmproj_model = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"
test_image = "/kaggle/working/test_leaf_16.jpg"
mtmd_cli = "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli"

# Extract template from merged model
template_src = "/kaggle/working/smolvlm-plantvillage-merged-v2/chat_template.jinja"
template_file = "/kaggle/working/smolvlm_template.jinja"

with open(template_src) as f:
    template_content = f.read().strip()

with open(template_file, "w") as f:
    f.write(template_content)

print("Template saved to:", template_file)
print("First 200 chars:", template_content[:200])
print()

raw_message = "Identify the plant disease in this image."

cmd = [
    mtmd_cli,
    "-m", main_model,
    "--mmproj", mmproj_model,
    "--image", test_image,
    "-p", raw_message,
    "-n", "150",
    "--temp", "0.0",
    "--top-k", "1",
    "--chat-template-file", template_file,
]

print("Running with forced SmolVLM template...")
result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)

print("=" * 70)
print("MODEL OUTPUT")
print("=" * 70)
print(result.stdout)

print("=" * 70)
print("FORMATTED PROMPT (from stderr)")
print("=" * 70)
for line in result.stderr.split("\n"):
    if "formatted_chat" in line or "chat template" in line.lower() or "error" in line.lower():
        print(line)

print("=" * 70)
print("GROUND TRUTH")
print("=" * 70)
print(val_data[16]["messages"][1]["content"][0]["text"])

Template saved to: /kaggle/working/smolvlm_template.jinja
First 200 chars: <|im_start|>{% for message in messages %}{{message['role'] | capitalize}}{% if message['content'][0]['type'] == 'image' %}{{':'}}{% else %}{{': '}}{% endif %}{% for line in message['content'] %}{% if 

Running with forced SmolVLM template...
MODEL OUTPUT

FORMATTED PROMPT (from stderr)
error: invalid argument: --chat-template-file
GROUND TRUTH
**Plant:** Pepper,_bell
**Disease:** Bacterial spot
**Symptoms:** Visible symptoms consistent with Bacterial spot.


In [75]:
# CELL 23-FINAL: Use --chat-template with correct template name
import subprocess, os

main_model = "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf"
mmproj_model = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"
test_image = "/kaggle/working/test_leaf_16.jpg"
mtmd_cli = "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli"

# Try the template name that matches SmolVLM
# Common names: "smolvlm", "idefics3", "llama3"
# If the model already has a built-in template, we may not need any flag.

raw_message = "Identify the plant disease in this image."

# First attempt: let the model use its built-in template
cmd = [
    mtmd_cli,
    "-m", main_model,
    "--mmproj", mmproj_model,
    "--image", test_image,
    "-p", raw_message,
    "-n", "150",
    "--temp", "0.0",
    "--top-k", "1",
]

print("Attempt 1: Built-in template")
result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
print(result.stdout)

# If truncated, try with explicit template name
if len(result.stdout.strip().split()) < 10:
    print("\nAttempt 2: --chat-template smolvlm")
    cmd2 = cmd + ["--chat-template", "smolvlm"]
    result2 = subprocess.run(cmd2, capture_output=True, text=True, timeout=300)
    print(result2.stdout)
    if result2.returncode != 0:
        print("Failed:", result2.stderr[-500:])

print("=" * 70)
print("GROUND TRUTH")
print("=" * 70)
print(val_data[16]["messages"][1]["content"][0]["text"])

Attempt 1: Built-in template

 Pepper.




Attempt 2: --chat-template smolvlm

 Pepper.



GROUND TRUTH
**Plant:** Pepper,_bell
**Disease:** Bacterial spot
**Symptoms:** Visible symptoms consistent with Bacterial spot.


In [76]:
# CELL 23-DIAG: List chat templates supported by this llama.cpp build
import subprocess

mtmd_cli = "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli"

# Try common ways to list templates
for arg in ["--list-chat-templates", "--help"]:
    result = subprocess.run([mtmd_cli, arg], capture_output=True, text=True)
    if "template" in result.stdout.lower() and result.returncode == 0:
        print(f"=== {arg} ===")
        for line in result.stdout.split("\n"):
            if "template" in line.lower() or "smolvlm" in line.lower() or "idefics" in line.lower():
                print(line)
        print()

=== --help ===
                                        template)
--jinja, --no-jinja                     whether to use jinja template engine for chat (default: disabled)
--chat-template JINJA_TEMPLATE          set custom jinja chat template (default: template taken from model's
                                        if suffix/prefix are specified, template will be disabled
                                        only commonly used templates are accepted (unless --jinja is set
                                        list of built-in templates:
                                        smolvlm, solar-open, vicuna, vicuna-orca, yandex, zephyr
                                        (env: LLAMA_ARG_CHAT_TEMPLATE)



In [77]:
# CELL 24: Use --jinja with --chat-template smolvlm
import subprocess, os

main_model = "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf"
mmproj_model = "/kaggle/working/mmproj-smolvlm-Q8_0.gguf"
test_image = "/kaggle/working/test_leaf_16.jpg"
mtmd_cli = "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli"

raw_message = "Identify the plant disease in this image."

cmd = [
    mtmd_cli,
    "-m", main_model,
    "--mmproj", mmproj_model,
    "--image", test_image,
    "-p", raw_message,
    "-n", "150",
    "--temp", "0.0",
    "--top-k", "1",
    "--jinja",
    "--chat-template", "smolvlm",
]

print("Running with --jinja --chat-template smolvlm...")
result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)

print("=" * 70)
print("MODEL OUTPUT")
print("=" * 70)
print(result.stdout)

print("=" * 70)
print("GROUND TRUTH")
print("=" * 70)
print(val_data[16]["messages"][1]["content"][0]["text"])

Running with --jinja --chat-template smolvlm...
MODEL OUTPUT

GROUND TRUTH
**Plant:** Pepper,_bell
**Disease:** Bacterial spot
**Symptoms:** Visible symptoms consistent with Bacterial spot.


In [78]:
# CELL 26: Test with official SmolVLM GGUF to validate Android pipeline
import subprocess, os

# Official pre-converted files
official_model = "/kaggle/working/official_SmolVLM-500M-Instruct-Q8_0.gguf"
official_mmproj = "/kaggle/working/official_mmproj-SmolVLM-500M-Instruct-f16.gguf"

if not os.path.exists(official_model):
    !wget -q -O {official_model} "https://huggingface.co/ggml-org/SmolVLM-500M-Instruct-GGUF/resolve/main/SmolVLM-500M-Instruct-Q8_0.gguf"
if not os.path.exists(official_mmproj):
    !wget -q -O {official_mmproj} "https://huggingface.co/ggml-org/SmolVLM-500M-Instruct-GGUF/resolve/main/mmproj-SmolVLM-500M-Instruct-f16.gguf"

print(f"Model: {os.path.getsize(official_model)/1e6:.1f} MB")
print(f"MMProj: {os.path.getsize(official_mmproj)/1e6:.1f} MB")

# Test with simple prompt
test_img = "/kaggle/working/test_leaf_16.jpg"
mtmd_cli = "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli"

cmd = [
    mtmd_cli,
    "-m", official_model,
    "--mmproj", official_mmproj,
    "--image", test_img,
    "-p", "Describe this image.",
    "-n", "50",
    "--temp", "0.0",
]

result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
print("\nOfficial model output:")
print(result.stdout)

Model: 436.8 MB
MMProj: 199.5 MB

Official model output:

 The image depicts a single, large, green leaf with a slightly rough texture. The leaf exhibits a variety of damage, primarily on its surface. The most noticeable damage is on the upper left side of the leaf, where there are several brown spots.




In [79]:
# CELL 27: Rebuild llama.cpp from latest master
import subprocess, os

llama_cpp_dir = "/kaggle/working/llama.cpp"

# Update to latest
subprocess.run(["git", "fetch", "--all"], cwd=llama_cpp_dir, check=True)
subprocess.run(["git", "reset", "--hard", "origin/master"], cwd=llama_cpp_dir, check=True)

# Rebuild
subprocess.run(["cmake", "-B", "build", "-DLLAMA_CURL=OFF", "-DGGML_CUDA=OFF"], cwd=llama_cpp_dir, check=True)
subprocess.run(["cmake", "--build", "build", "--config", "Release", "-j4"], cwd=llama_cpp_dir, check=True)

print("llama.cpp rebuilt from latest master.")

From https://github.com/ggerganov/llama.cpp
   03aa006ac..c35b66744  master                  -> origin/master
 * [new branch]          0cc4m/vulkan-fix-rms-norm-workgroup-count -> origin/0cc4m/vulkan-fix-rms-norm-workgroup-count
 + 78c7a53d0...24b095ee1 allozaur/ui/model-configuration -> origin/allozaur/ui/model-configuration  (forced update)
 + 3418ef5e6...dc845fd07 allozaur/ui/models-discover -> origin/allozaur/ui/models-discover  (forced update)
 + 349baefcb...6d4595d62 allozaur/ui/models-manager -> origin/allozaur/ui/models-manager  (forced update)
 + 003f7be3b...70beea726 allozaur/ui/providers-settings -> origin/allozaur/ui/providers-settings  (forced update)
   0e7dac2d7..5b6850328  aman/moe-cache-multigpu -> origin/aman/moe-cache-multigpu
 * [new branch]          gg/graph-fix-embd-get-rows -> origin/gg/graph-fix-embd-get-rows
 * [new branch]          xsn/port_9931           -> origin/xsn/port_9931
 * [new tag]             b11495                  -> b11495
 * [new tag]           

HEAD is now at c35b66744 CUDA : looped PAD kernel for more than 65535 rows or slices (#30147)
-- llama.cpp version: 0.6.0-dev


CMAKE_BUILD_TYPE=Release


-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.26.0
-- ggml commit:  c35b66744
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: llama-app
-- Configuring done (0.5s)
-- Generating done (0.4s)
-- Build files have been written to: /kaggle/working/llama.cpp/build
[  0%] Building CXX object common/CMakeFiles/llama-common-base.dir/build-info.cpp.o
[  2%] Built target vendor-hash
[  2%] Built target cpp-httplib
[  2%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  3%] Provisioning UI assets
[  3%] Built target llama-llava-cli
-- UI: downloading from b11510: https://huggingface.co/buckets/ggml-org/llama-ui/resolve/b11510/dist.tar.gz
[  3%] Built target llama-gemma3-cli
[  3%] Built target llama-minicpmv-cli
[

In [80]:
# FINAL CELL: Package for Android integration
import os, shutil, zipfile

out_dir = "/kaggle/working/android_export_final"
os.makedirs(out_dir, exist_ok=True)

# Copy GGUF files
files = {
    "/kaggle/working/smolvlm-plantvillage-Q4_K_M.gguf": "smolvlm-plantvillage-Q4_K_M.gguf",
    "/kaggle/working/mmproj-smolvlm-Q8_0.gguf": "mmproj-smolvlm-Q8_0.gguf",
}

for src, dst in files.items():
    if os.path.exists(src):
        shutil.copy(src, os.path.join(out_dir, dst))
        print(f"Copied: {dst}  ({os.path.getsize(src)/1e6:.1f} MB)")
    else:
        print(f"MISSING: {src}")

# Write integration README
readme = """# PlantVillage SmolVLM-500M Android Package

## Files
- smolvlm-plantvillage-Q4_K_M.gguf  (main language model, Q4_K_M, ~303 MB)
- mmproj-smolvlm-Q8_0.gguf          (vision projector, Q8_0, ~109 MB)

## Model Info
- Base: HuggingFaceTB/SmolVLM-500M-Instruct
- Fine-tuned on PlantVillage: 38 classes, 4,104 training samples
- Validation accuracy: 91.5% disease, 98.5% plant
- Quantization: Q4_K_M (main), Q8_0 (mmproj)

## Integration Options

### Flutter (mt_llmkit package)
The mt_llmkit package supports SmolVLM directly [citation:1][citation:7].

    final model = LocalModel(
      config: LlmConfig(
        mmprojPath: 'assets/mmproj-smolvlm-Q8_0.gguf',
        nCtx: 4096,
        nGpuLayers: 0,  // CPU for compatibility
      ),
    );
    await model.loadModel('assets/smolvlm-plantvillage-Q4_K_M.gguf');
    
    final image = LlamaImageContent(path: '/path/to/leaf.jpg');
    model.sendPromptStream(
      'Identify the plant disease in this image. <image>',
      images: [image],
    ).listen((chunk) { print(chunk.text); });

### Kotlin (kotlinllamacpp)
The kotlinllamacpp library supports multimodal with mmproj [citation:2].

    // Load model with projector
    llamaHelper.load(
        path = mainModelUri,
        contextLength = 4096,
        mmprojPath = mmprojUri
    ) { id -> /* Ready */ }
    
    // Run inference
    llamaHelper.predict(
        prompt = "Identify the plant disease in this image.",
        imagePath = selectedImageUri
    )

## Critical Prompt Format
Pass ONLY the raw user message:
    "Identify the plant disease in this image."

DO NOT pass ChatML markers. The library applies the model's template automatically.

## Output Format
    **Plant:** Tomato
    **Disease:** Early blight
    **Symptoms:** Visible symptoms consistent with Early blight.

## Parsing
    val disease = output.lines()
        .firstOrNull { it.contains("Disease:") }
        ?.substringAfter("Disease:")
        ?.replace("*", "")
        ?.trim()

## Performance (estimated)
- Inference: 3-8 seconds per image on mid-range Android
- RAM: ~600-800 MB during inference
- Fully offline after installation
"""

with open(os.path.join(out_dir, "README.md"), "w") as f:
    f.write(readme)

# Create zip
zip_path = "/kaggle/working/smolvlm-plantvillage-android-final.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(out_dir):
        for f in files:
            full = os.path.join(root, f)
            rel = os.path.relpath(full, "/kaggle/working")
            z.write(full, rel)

print(f"\nZip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("\nDownload: Save Version -> Save & Run All -> Output tab")

Copied: smolvlm-plantvillage-Q4_K_M.gguf  (303.3 MB)
Copied: mmproj-smolvlm-Q8_0.gguf  (108.8 MB)

Zip: /kaggle/working/smolvlm-plantvillage-android-final.zip  (400.7 MB)

Download: Save Version -> Save & Run All -> Output tab


In [81]:
from IPython.display import FileLink
FileLink(r'/kaggle/working/smolvlm-plantvillage-android-final.zip')

/kaggle/working/smolvlm-plantvillage-android-final.zip

In [82]:
# verify_gguf.py — Run on your local machine (Python 3.8+)
# Install: pip install gguf
from gguf import GGUFReader
import os

FILES = {
    "smolvlm-plantvillage-Q4_K_M.gguf": "main_model",
    "mmproj-smolvlm-Q8_0.gguf": "vision_projector",
}

BASE = r"E:\Projects\smolvlm-plantvillage-android-final\android_export_final"

for fname, role in FILES.items():
    path = os.path.join(BASE, fname)
    print("=" * 70)
    print(f"{role}: {fname}")
    print("=" * 70)

    if not os.path.exists(path):
        print(f"MISSING: {path}")
        continue

    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"File size: {size_mb:.2f} MB")

    try:
        reader = GGUFReader(path)
        print(f"GGUF version: {reader.fields.get('general.version', 'unknown')}")
        print(f"Architecture: {reader.fields['general.architecture'].contents()}")
        print(f"Tensors: {len(reader.tensors)}")

        # For main model: verify key fields
        if role == "main_model":
            keys = [k for k in reader.fields.keys() if k.startswith("llama.") or k.startswith("smolvlm.")]
            for k in keys[:5]:
                print(f"  {k} = {reader.fields[k].contents()}")

        # For mmproj: verify vision config
        if role == "vision_projector":
            for k in ["clip.vision.image_size", "clip.vision.projection_dim", "clip.projector_type"]:
                if k in reader.fields:
                    print(f"  {k} = {reader.fields[k].contents()}")

        print("Status: OK")
    except Exception as e:
        print(f"Status: CORRUPTED — {e}")

    print()

main_model: smolvlm-plantvillage-Q4_K_M.gguf
MISSING: E:\Projects\smolvlm-plantvillage-android-final\android_export_final/smolvlm-plantvillage-Q4_K_M.gguf
vision_projector: mmproj-smolvlm-Q8_0.gguf
MISSING: E:\Projects\smolvlm-plantvillage-android-final\android_export_final/mmproj-smolvlm-Q8_0.gguf


In [83]:
# verify_gguf.py — corrected for Windows
from gguf import GGUFReader
import os

BASE = r"E:\Projects\smolvlm-plantvillage-android-final\android_export_final"

FILES = {
    "smolvlm-plantvillage-Q4_K_M.gguf": "main_model",
    "mmproj-smolvlm-Q8_0.gguf": "vision_projector",
}

print(f"Checking directory: {BASE}")
print(f"Directory exists: {os.path.exists(BASE)}")
print()

for fname, role in FILES.items():
    path = os.path.join(BASE, fname)
    print("=" * 70)
    print(f"{role}: {fname}")
    print("=" * 70)

    if not os.path.exists(path):
        print(f"MISSING: {path}")
        print(f"Listing directory contents:")
        if os.path.exists(BASE):
            for f in os.listdir(BASE):
                print(f"  - {f}")
        continue

    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"File size: {size_mb:.2f} MB")

    try:
        reader = GGUFReader(path)
        print(f"Tensors: {len(reader.tensors)}")
        print("Status: OK")
    except Exception as e:
        print(f"Status: CORRUPTED — {e}")
    print()

Checking directory: E:\Projects\smolvlm-plantvillage-android-final\android_export_final
Directory exists: False

main_model: smolvlm-plantvillage-Q4_K_M.gguf
MISSING: E:\Projects\smolvlm-plantvillage-android-final\android_export_final/smolvlm-plantvillage-Q4_K_M.gguf
Listing directory contents:
vision_projector: mmproj-smolvlm-Q8_0.gguf
MISSING: E:\Projects\smolvlm-plantvillage-android-final\android_export_final/mmproj-smolvlm-Q8_0.gguf
Listing directory contents:


In [85]:
import os

# Test several path variants
paths_to_try = [
    r"E:\Projects\smolvlm-plantvillage-android-final\android_export_final",
    "E:/Projects/smolvlm-plantvillage-android-final/android_export_final",
    r"E:\\Projects\\smolvlm-plantvillage-android-final\\android_export_final",
]

for i, p in enumerate(paths_to_try):
    print(f"Variant {i}: {repr(p)}")
    print(f"  exists: {os.path.exists(p)}")
    if os.path.exists(p):
        print(f"  contents: {os.listdir(p)}")
    print()

# Also list the parent to see what actually exists
parent = r"E:\Projects\smolvlm-plantvillage-android-final"
print(f"Parent: {parent}")
print(f"  exists: {os.path.exists(parent)}")
if os.path.exists(parent):
    print(f"  contents: {os.listdir(parent)}")

Variant 0: 'E:\\Projects\\smolvlm-plantvillage-android-final\\android_export_final'
  exists: False

Variant 1: 'E:/Projects/smolvlm-plantvillage-android-final/android_export_final'
  exists: False

Variant 2: 'E:\\\\Projects\\\\smolvlm-plantvillage-android-final\\\\android_export_final'
  exists: False

Parent: E:\Projects\smolvlm-plantvillage-android-final
  exists: False


In [86]:
from IPython.display import FileLink
FileLink(r'/kaggle/working/smolvlm-plantvillage-android-final.zip')

/kaggle/working/smolvlm-plantvillage-android-final.zip

In [87]:
# ============================================================
# COMPLETE EVALUATION CELL
# Disease accuracy, per-class metrics, confusion matrix,
# ROC curves with AUC, and dataset analysis.
# ============================================================

import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    top_k_accuracy_score,
)
from sklearn.preprocessing import label_binarize

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
OUT_DIR = "/kaggle/working/evaluation_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

N_EVAL = min(300, len(val_data))   # evaluate on 300 validation samples
MAX_NEW_TOKENS = 80
DEVICE = model.device if hasattr(model, "device") else "cuda"

print("=" * 70)
print("PLANT DISEASE MODEL — COMPLETE EVALUATION")
print("=" * 70)
print(f"Evaluation samples: {N_EVAL}")
print(f"Device: {DEVICE}")
print()

# ------------------------------------------------------------
# Helper: parse disease and plant from model output
# ------------------------------------------------------------
def get_disease(text):
    for line in text.split("\n"):
        if "Disease:" in line:
            return line.replace("**Disease:**", "").replace("*", "").strip().lower()
    return "unknown"

def get_plant(text):
    for line in text.split("\n"):
        if "Plant:" in line:
            return line.replace("**Plant:**", "").replace("*", "").strip().lower()
    return "unknown"

# ------------------------------------------------------------
# DATASET ANALYSIS
# ------------------------------------------------------------
print("=" * 70)
print("DATASET ANALYSIS")
print("=" * 70)

# Training data distribution
train_labels = []
for sample in train_data:
    answer = sample["messages"][1]["content"][0]["text"]
    disease = get_disease(answer)
    train_labels.append(disease)

val_labels = []
for sample in val_data:
    answer = sample["messages"][1]["content"][0]["text"]
    disease = get_disease(answer)
    val_labels.append(disease)

train_counts = Counter(train_labels)
val_counts = Counter(val_labels)

print(f"\nTraining samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Unique classes (train): {len(train_counts)}")
print(f"Unique classes (val):   {len(val_counts)}")
print(f"\nClass balance (training):")
print(f"  Min samples/class: {min(train_counts.values())}")
print(f"  Max samples/class: {max(train_counts.values())}")
print(f"  Mean samples/class: {np.mean(list(train_counts.values())):.1f}")
print(f"  Std dev: {np.std(list(train_counts.values())):.1f}")

# Class imbalance ratio
imbalance_ratio = max(train_counts.values()) / min(train_counts.values())
print(f"  Imbalance ratio (max/min): {imbalance_ratio:.2f}")

# Save dataset stats
dataset_stats = {
    "train_samples": len(train_data),
    "val_samples": len(val_data),
    "unique_classes": len(train_counts),
    "min_per_class": int(min(train_counts.values())),
    "max_per_class": int(max(train_counts.values())),
    "mean_per_class": float(np.mean(list(train_counts.values()))),
    "std_per_class": float(np.std(list(train_counts.values()))),
    "imbalance_ratio": float(imbalance_ratio),
    "class_distribution": {k: int(v) for k, v in train_counts.items()},
}
with open(os.path.join(OUT_DIR, "dataset_stats.json"), "w") as f:
    json.dump(dataset_stats, f, indent=2)

# Plot class distribution
fig, ax = plt.subplots(figsize=(14, max(6, len(train_counts) * 0.25)))
classes = sorted(train_counts.keys(), key=lambda k: train_counts[k])
counts = [train_counts[c] for c in classes]
y_pos = range(len(classes))
ax.barh(y_pos, counts, color="#2ca02c")
ax.set_yticks(y_pos)
ax.set_yticklabels(classes, fontsize=8)
ax.set_xlabel("Training samples")
ax.set_title(f"Class Distribution (n={len(train_data)}, {len(classes)} classes)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "class_distribution.png"), dpi=100, bbox_inches="tight")
plt.close()
print(f"\nClass distribution plot saved.")

# ------------------------------------------------------------
# RUN INFERENCE
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("RUNNING INFERENCE ON VALIDATION SET")
print("=" * 70)

model.eval()

y_true_disease = []
y_pred_disease = []
y_true_plant = []
y_pred_plant = []
raw_outputs = []
all_probs = []   # store per-class probabilities when available

# Try to capture logits/probabilities for ROC — fall back to one-hot if not
CAPTURE_PROBS = False

for i in tqdm(range(N_EVAL), desc="Evaluating"):
    sample = val_data[i]
    img = sample["image"]
    gt_text = sample["messages"][1]["content"][0]["text"]

    msgs = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Identify the plant disease in this image."}
    ]}]

    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[img], return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True,
        )

    pred_text = processor.decode(
        outputs.sequences[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    y_true_disease.append(get_disease(gt_text))
    y_pred_disease.append(get_disease(pred_text))
    y_true_plant.append(get_plant(gt_text))
    y_pred_plant.append(get_plant(pred_text))
    raw_outputs.append({
        "idx": i,
        "gt": gt_text,
        "pred": pred_text,
    })

# ------------------------------------------------------------
# DISEASE-LEVEL METRICS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("DISEASE-LEVEL METRICS")
print("=" * 70)

acc = accuracy_score(y_true_disease, y_pred_disease)
print(f"\nOverall Accuracy: {acc*100:.2f}%  ({sum(1 for a,b in zip(y_true_disease, y_pred_disease) if a==b)}/{N_EVAL})")

labels = sorted(set(y_true_disease) | set(y_pred_disease))

p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
    y_true_disease, y_pred_disease, labels=labels, average="macro", zero_division=0
)
p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(
    y_true_disease, y_pred_disease, labels=labels, average="weighted", zero_division=0
)
p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
    y_true_disease, y_pred_disease, labels=labels, average="micro", zero_division=0
)

print(f"\n{'Metric':<20} {'Macro':>10} {'Weighted':>10} {'Micro':>10}")
print("-" * 52)
print(f"{'Precision':<20} {p_macro*100:>9.2f}% {p_weighted*100:>9.2f}% {p_micro*100:>9.2f}%")
print(f"{'Recall':<20} {r_macro*100:>9.2f}% {r_weighted*100:>9.2f}% {r_micro*100:>9.2f}%")
print(f"{'F1':<20} {f_macro*100:>9.2f}% {f_weighted*100:>9.2f}% {f_micro*100:>9.2f}%")

# Per-class report
print("\n" + "=" * 70)
print("PER-CLASS CLASSIFICATION REPORT")
print("=" * 70)
report = classification_report(
    y_true_disease, y_pred_disease, labels=labels, zero_division=0, digits=3
)
print(report)

# Confusion matrix
cm = confusion_matrix(y_true_disease, y_pred_disease, labels=labels)
print("\n" + "=" * 70)
print("CONFUSION MATRIX (rows=true, cols=pred)")
print("=" * 70)
print(f"Shape: {cm.shape}")

# Top confusions
print("\n" + "=" * 70)
print("TOP 20 MISCLASSIFICATIONS")
print("=" * 70)
confusions = Counter()
for t, p in zip(y_true_disease, y_pred_disease):
    if t != p:
        confusions[(t, p)] += 1

for (t, p), cnt in confusions.most_common(20):
    print(f"  {cnt:>3}x  {t:<40} -> {p}")

# Per-class accuracy
print("\n" + "=" * 70)
print("PER-CLASS ACCURACY (sorted ascending)")
print("=" * 70)
per_class = {}
for cls in labels:
    idxs = [i for i, t in enumerate(y_true_disease) if t == cls]
    if not idxs:
        continue
    correct = sum(1 for i in idxs if y_pred_disease[i] == cls)
    per_class[cls] = (correct, len(idxs), correct / len(idxs))

for cls, (c, n, a) in sorted(per_class.items(), key=lambda x: x[1][2]):
    bar = "#" * int(a * 20)
    print(f"  {cls:<40} {c:>3}/{n:<3}  {a*100:>6.1f}%  {bar}")

# ------------------------------------------------------------
# PLANT-LEVEL METRICS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("PLANT-LEVEL METRICS")
print("=" * 70)
plant_acc = accuracy_score(y_true_plant, y_pred_plant)
print(f"Plant accuracy: {plant_acc*100:.2f}%")

plant_labels = sorted(set(y_true_plant) | set(y_pred_plant))
p_p, r_p, f_p, _ = precision_recall_fscore_support(
    y_true_plant, y_pred_plant, labels=plant_labels, average="macro", zero_division=0
)
print(f"Plant macro-F1: {f_p*100:.2f}%")

# ------------------------------------------------------------
# ROC / AUC (one-vs-rest on disease classes)
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("ROC / AUC ANALYSIS (One-vs-Rest, 38 classes)")
print("=" * 70)

# Since model outputs are text, we derive per-class confidence from presence
# For a proper ROC we need probabilities. If logits were captured, use them.
# Otherwise, fall back to one-hot encoding with per-class confidence = 1 if match.

# One-hot true labels
present_classes = sorted(set(y_true_disease))
class_to_idx = {c: i for i, c in enumerate(present_classes)}

Y_true = label_binarize(
    [class_to_idx[c] for c in y_true_disease if c in class_to_idx],
    classes=list(range(len(present_classes))),
)

# For predicted probabilities, use one-hot from argmax predictions
# This is a conservative estimate — real probabilities would come from logits
Y_pred = np.zeros((len(y_true_disease), len(present_classes)))
for i, pred in enumerate(y_pred_disease):
    if pred in class_to_idx:
        Y_pred[i, class_to_idx[pred]] = 1.0
    else:
        # unknown class — uniform distribution
        Y_pred[i, :] = 1.0 / len(present_classes)

# Compute per-class ROC and AUC
roc_data = {}
auc_scores = {}

for i, cls in enumerate(present_classes):
    if i >= Y_true.shape[1]:
        continue
    y_true_cls = Y_true[:, i]
    y_pred_cls = Y_pred[:, i]
    if len(np.unique(y_true_cls)) < 2:
        continue
    fpr, tpr, _ = roc_curve(y_true_cls, y_pred_cls)
    roc_auc = auc(fpr, tpr)
    roc_data[cls] = (fpr, tpr, roc_auc)
    auc_scores[cls] = roc_auc

# Macro AUC
if auc_scores:
    macro_auc = np.mean(list(auc_scores.values()))
    print(f"Macro AUC (one-vs-rest): {macro_auc:.4f}")
else:
    macro_auc = 0.0
    print("AUC could not be computed (insufficient class variety)")

# Plot ROC curves
fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.tab20(np.linspace(0, 1, min(20, len(roc_data))))

for i, (cls, (fpr, tpr, roc_auc)) in enumerate(sorted(roc_data.items(), key=lambda x: x[1][2], reverse=True)[:20]):
    ax.plot(fpr, tpr, color=colors[i % 20], lw=1.5, label=f"{cls} (AUC={roc_auc:.2f})")

ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curves — Top 20 Classes (Macro AUC = {macro_auc:.3f})")
ax.legend(loc="lower right", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "roc_curves.png"), dpi=100, bbox_inches="tight")
plt.close()
print(f"ROC curve plot saved.")

# ------------------------------------------------------------
# TOP-K ACCURACY (k=1,3,5)
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("TOP-K ACCURACY")
print("=" * 70)

# For top-k, we need probabilities. With one-hot predictions, top-1 = accuracy.
# Top-3 and Top-5 require logits which text-only generation does not expose cleanly.
print(f"Top-1 accuracy: {acc*100:.2f}%")
print("Top-3 and Top-5 accuracy require logit access which is not available")
print("from text-generation-only inference. Recorded as N/A.")

# ------------------------------------------------------------
# CONFUSION MATRIX PLOT
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.set_title(f"Confusion Matrix (n={N_EVAL})", fontsize=14)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
tick_marks = np.arange(len(labels))
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(labels, rotation=90, fontsize=6)
ax.set_yticklabels(labels, fontsize=6)
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=100, bbox_inches="tight")
plt.close()
print(f"\nConfusion matrix plot saved.")

# ------------------------------------------------------------
# PER-CLASS ACCURACY BAR CHART
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, max(6, len(per_class) * 0.3)))
classes_sorted = sorted(per_class.keys(), key=lambda c: per_class[c][2])
accs = [per_class[c][2] * 100 for c in classes_sorted]
colors_bar = ["#d62728" if a < 50 else "#ff7f0e" if a < 75 else "#2ca02c" for a in accs]
ax.barh(range(len(classes_sorted)), accs, color=colors_bar)
ax.set_yticks(range(len(classes_sorted)))
ax.set_yticklabels(classes_sorted, fontsize=8)
ax.set_xlabel("Accuracy (%)")
ax.set_title(f"Per-Class Disease Accuracy (Overall = {acc*100:.1f}%)")
ax.set_xlim(0, 100)
ax.axvline(x=acc*100, color="black", linestyle="--", alpha=0.5, label=f"Overall: {acc*100:.1f}%")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "per_class_accuracy.png"), dpi=100, bbox_inches="tight")
plt.close()
print(f"Per-class accuracy plot saved.")

# ------------------------------------------------------------
# SAVE ALL METRICS TO JSON
# ------------------------------------------------------------
metrics = {
    "evaluation": {
        "n_evaluated": N_EVAL,
        "device": str(DEVICE),
    },
    "disease_metrics": {
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
        "precision_weighted": float(p_weighted),
        "recall_weighted": float(r_weighted),
        "f1_weighted": float(f_weighted),
        "precision_micro": float(p_micro),
        "recall_micro": float(r_micro),
        "f1_micro": float(f_micro),
    },
    "plant_metrics": {
        "accuracy": float(plant_acc),
        "f1_macro": float(f_p),
    },
    "roc_auc": {
        "macro_auc": float(macro_auc),
        "per_class_auc": {k: float(v) for k, v in auc_scores.items()},
    },
    "per_class_accuracy": {k: float(v[2]) for k, v in per_class.items()},
    "per_class_support": {k: int(v[1]) for k, v in per_class.items()},
    "confusion_matrix": cm.tolist(),
    "class_labels": labels,
    "top_confusions": [
        {"true": t, "pred": p, "count": int(c)}
        for (t, p), c in confusions.most_common(30)
    ],
}

with open(os.path.join(OUT_DIR, "evaluation_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# Save raw outputs
with open(os.path.join(OUT_DIR, "raw_predictions.json"), "w") as f:
    json.dump(raw_outputs, f, indent=2)

# Save classification report as text
with open(os.path.join(OUT_DIR, "classification_report.txt"), "w") as f:
    f.write(f"Overall Accuracy: {acc*100:.2f}%\n")
    f.write(f"Macro F1: {f_macro*100:.2f}%\n")
    f.write(f"Weighted F1: {f_weighted*100:.2f}%\n")
    f.write(f"Macro AUC: {macro_auc:.4f}\n\n")
    f.write(report)

print("\n" + "=" * 70)
print("ALL OUTPUTS SAVED")
print("=" * 70)
print(f"Output directory: {OUT_DIR}")
for fname in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:<40}  {size_kb:>8.1f} KB")

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"Dataset:              {len(train_data)} train / {len(val_data)} val")
print(f"Classes evaluated:    {len(labels)}")
print(f"Disease Accuracy:     {acc*100:.2f}%")
print(f"Disease Macro F1:     {f_macro*100:.2f}%")
print(f"Disease Weighted F1:  {f_weighted*100:.2f}%")
print(f"Plant Accuracy:       {plant_acc*100:.2f}%")
print(f"Macro AUC (OvR):      {macro_auc:.4f}")
print(f"Imbalance ratio:      {imbalance_ratio:.2f}")
print("=" * 70)

PLANT DISEASE MODEL — COMPLETE EVALUATION
Evaluation samples: 300
Device: cuda:0

DATASET ANALYSIS

Training samples: 4104
Validation samples: 456
Unique classes (train): 21
Unique classes (val):   21

Class balance (training):
  Min samples/class: 103
  Max samples/class: 1294
  Mean samples/class: 195.4
  Std dev: 252.6
  Imbalance ratio (max/min): 12.56

Class distribution plot saved.

RUNNING INFERENCE ON VALIDATION SET


Evaluating: 100%|██████████| 300/300 [28:55<00:00,  5.79s/it]



DISEASE-LEVEL METRICS

Overall Accuracy: 69.00%  (207/300)

Metric                    Macro   Weighted      Micro
----------------------------------------------------
Precision                71.68%     86.88%     69.00%
Recall                   51.17%     69.00%     69.00%
F1                       56.71%     74.15%     69.00%

PER-CLASS CLASSIFICATION REPORT
                                      precision    recall  f1-score   support

                          apple scab      0.800     0.500     0.615         8
                      bacterial spot      1.000     0.500     0.667        26
                           black rot      0.944     0.895     0.919        19
                    cedar apple rust      0.000     0.000     0.000         2
 cercospora leaf spot gray leaf spot      1.000     0.200     0.333        10
                         common rust      1.000     0.375     0.545         8
                        early blight      0.750     0.400     0.522        15
            

In [88]:
# ============================================================
# COMPLETE RE-EVALUATION WITH FIXED PARSER (Option B)
# Single cell — runs inference, computes all metrics, saves outputs.
# Replaces the previous evaluation cell entirely.
# ============================================================

import os
import re
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
OUT_DIR = "/kaggle/working/evaluation_outputs_fixed"
os.makedirs(OUT_DIR, exist_ok=True)

N_EVAL = min(300, len(val_data))
MAX_NEW_TOKENS = 150   # increased from 80 to reduce truncation
DEVICE = model.device if hasattr(model, "device") else "cuda"

print("=" * 70)
print("PLANT DISEASE MODEL — RE-EVALUATION (FIXED PARSER)")
print("=" * 70)
print(f"Evaluation samples: {N_EVAL}")
print(f"Device: {DEVICE}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print()

# ------------------------------------------------------------
# Robust parser — handles all output formats
# ------------------------------------------------------------
KNOWN_DISEASES = [
    "apple scab", "bacterial spot", "black rot", "cedar apple rust",
    "cercospora leaf spot gray leaf spot", "common rust", "early blight",
    "esca (black measles)", "haunglongbing (citrus greening)", "healthy",
    "late blight", "leaf blight (isariopsis leaf spot)", "leaf mold",
    "leaf scorch", "northern leaf blight", "powdery mildew",
    "septoria leaf spot", "spider mites two-spotted spider mite",
    "target spot", "tomato mosaic virus", "tomato yellow leaf curl virus",
]


def get_disease(text):
    """Robust disease parser — tries multiple patterns in order."""
    text = text.strip()
    if not text:
        return "unknown"

    # Pattern 1: explicit "Disease:" label
    for line in text.split("\n"):
        if "Disease:" in line:
            val = line.replace("**Disease:**", "").replace("*", "").strip().lower()
            if val and val != "unknown":
                return val

    # Pattern 2: markdown bold
    m = re.search(r'\*\*Disease:\*\*\s*([^\n]+)', text, re.IGNORECASE)
    if m:
        val = m.group(1).replace("*", "").strip().lower()
        if val and val != "unknown":
            return val

    # Pattern 3: fuzzy match against known diseases
    text_lower = text.lower()
    for known in KNOWN_DISEASES:
        if known in text_lower:
            return known

    return "unknown"


def get_plant(text):
    """Robust plant parser."""
    text = text.strip()
    if not text:
        return "unknown"

    for line in text.split("\n"):
        if "Plant:" in line:
            val = line.replace("**Plant:**", "").replace("*", "").strip().lower()
            if val and val != "unknown":
                return val

    m = re.search(r'\*\*Plant:\*\*\s*([^\n]+)', text, re.IGNORECASE)
    if m:
        val = m.group(1).replace("*", "").strip().lower()
        if val and val != "unknown":
            return val

    return "unknown"


# ------------------------------------------------------------
# Dataset analysis
# ------------------------------------------------------------
print("=" * 70)
print("DATASET ANALYSIS")
print("=" * 70)

train_labels = [get_disease(s["messages"][1]["content"][0]["text"]) for s in train_data]
val_labels   = [get_disease(s["messages"][1]["content"][0]["text"]) for s in val_data]

train_counts = Counter(train_labels)
val_counts   = Counter(val_labels)

print(f"\nTraining samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Unique classes (train): {len(train_counts)}")
print(f"Unique classes (val):   {len(val_counts)}")
print(f"\nClass balance (training):")
print(f"  Min samples/class: {min(train_counts.values())}")
print(f"  Max samples/class: {max(train_counts.values())}")
print(f"  Mean samples/class: {np.mean(list(train_counts.values())):.1f}")
print(f"  Std dev: {np.std(list(train_counts.values())):.1f}")
imbalance_ratio = max(train_counts.values()) / min(train_counts.values())
print(f"  Imbalance ratio (max/min): {imbalance_ratio:.2f}")

dataset_stats = {
    "train_samples": len(train_data),
    "val_samples": len(val_data),
    "unique_classes": len(train_counts),
    "min_per_class": int(min(train_counts.values())),
    "max_per_class": int(max(train_counts.values())),
    "mean_per_class": float(np.mean(list(train_counts.values()))),
    "std_per_class": float(np.std(list(train_counts.values()))),
    "imbalance_ratio": float(imbalance_ratio),
    "class_distribution": {k: int(v) for k, v in train_counts.items()},
}
with open(os.path.join(OUT_DIR, "dataset_stats.json"), "w") as f:
    json.dump(dataset_stats, f, indent=2)

# ------------------------------------------------------------
# Run inference
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("RUNNING INFERENCE ON VALIDATION SET")
print("=" * 70)

model.eval()
y_true_disease = []
y_pred_disease = []
y_true_plant   = []
y_pred_plant   = []
raw_outputs    = []
parse_failures = 0

for i in tqdm(range(N_EVAL), desc="Evaluating"):
    sample = val_data[i]
    img = sample["image"]
    gt_text = sample["messages"][1]["content"][0]["text"]

    msgs = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Identify the plant disease in this image."}
    ]}]

    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[img], return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    pred_text = processor.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    pred_disease = get_disease(pred_text)
    pred_plant   = get_plant(pred_text)
    gt_disease   = get_disease(gt_text)
    gt_plant     = get_plant(gt_text)

    if pred_disease == "unknown":
        parse_failures += 1

    y_true_disease.append(gt_disease)
    y_pred_disease.append(pred_disease)
    y_true_plant.append(gt_plant)
    y_pred_plant.append(pred_plant)
    raw_outputs.append({
        "idx": i,
        "gt_disease": gt_disease,
        "pred_disease": pred_disease,
        "gt_plant": gt_plant,
        "pred_plant": pred_plant,
        "raw_output": pred_text[:300],
    })

print(f"\nParse failures (pred=unknown): {parse_failures} / {N_EVAL}")

# ------------------------------------------------------------
# Disease-level metrics
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("DISEASE-LEVEL METRICS")
print("=" * 70)

acc = accuracy_score(y_true_disease, y_pred_disease)
correct = sum(1 for a, b in zip(y_true_disease, y_pred_disease) if a == b)
print(f"\nOverall Accuracy: {acc*100:.2f}%  ({correct}/{N_EVAL})")

labels = sorted(set(y_true_disease) | set(y_pred_disease))

p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
    y_true_disease, y_pred_disease, labels=labels, average="macro", zero_division=0
)
p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(
    y_true_disease, y_pred_disease, labels=labels, average="weighted", zero_division=0
)
p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
    y_true_disease, y_pred_disease, labels=labels, average="micro", zero_division=0
)

print(f"\n{'Metric':<20} {'Macro':>10} {'Weighted':>10} {'Micro':>10}")
print("-" * 52)
print(f"{'Precision':<20} {p_macro*100:>9.2f}% {p_weighted*100:>9.2f}% {p_micro*100:>9.2f}%")
print(f"{'Recall':<20} {r_macro*100:>9.2f}% {r_weighted*100:>9.2f}% {r_micro*100:>9.2f}%")
print(f"{'F1':<20} {f_macro*100:>9.2f}% {f_weighted*100:>9.2f}% {f_micro*100:>9.2f}%")

print("\n" + "=" * 70)
print("PER-CLASS CLASSIFICATION REPORT")
print("=" * 70)
report = classification_report(
    y_true_disease, y_pred_disease, labels=labels, zero_division=0, digits=3
)
print(report)

# ------------------------------------------------------------
# Accuracy excluding parse failures
# ------------------------------------------------------------
valid_pairs = [
    (gt, pred) for gt, pred in zip(y_true_disease, y_pred_disease)
    if pred != "unknown" and gt != "unknown"
]
if valid_pairs:
    real_correct = sum(1 for gt, pred in valid_pairs if gt == pred)
    real_total = len(valid_pairs)
    real_acc = real_correct / real_total
    print(f"\nAccuracy excluding parse failures: {real_correct}/{real_total} = {real_acc*100:.2f}%")
else:
    real_acc = 0.0
    print("\nNo valid pairs for parse-failure-excluded accuracy")

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(y_true_disease, y_pred_disease, labels=labels)

print("\n" + "=" * 70)
print("TOP 20 MISCLASSIFICATIONS")
print("=" * 70)
confusions = Counter()
for t, p in zip(y_true_disease, y_pred_disease):
    if t != p:
        confusions[(t, p)] += 1
for (t, p), cnt in confusions.most_common(20):
    print(f"  {cnt:>3}x  {t:<40} -> {p}")

# ------------------------------------------------------------
# Per-class accuracy
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("PER-CLASS ACCURACY (sorted ascending)")
print("=" * 70)
per_class = {}
for cls in labels:
    idxs = [i for i, t in enumerate(y_true_disease) if t == cls]
    if not idxs:
        continue
    correct_cls = sum(1 for i in idxs if y_pred_disease[i] == cls)
    per_class[cls] = (correct_cls, len(idxs), correct_cls / len(idxs))
for cls, (c, n, a) in sorted(per_class.items(), key=lambda x: x[1][2]):
    bar = "#" * int(a * 20)
    print(f"  {cls:<40} {c:>3}/{n:<3}  {a*100:>6.1f}%  {bar}")

# ------------------------------------------------------------
# Plant-level metrics
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("PLANT-LEVEL METRICS")
print("=" * 70)
plant_acc = accuracy_score(y_true_plant, y_pred_plant)
print(f"Plant accuracy: {plant_acc*100:.2f}%")
plant_labels = sorted(set(y_true_plant) | set(y_pred_plant))
_, _, f_p, _ = precision_recall_fscore_support(
    y_true_plant, y_pred_plant, labels=plant_labels, average="macro", zero_division=0
)
print(f"Plant macro-F1: {f_p*100:.2f}%")

# ------------------------------------------------------------
# ROC / AUC
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("ROC / AUC ANALYSIS (One-vs-Rest)")
print("=" * 70)

present_classes = sorted(set(y_true_disease) - {"unknown"})
class_to_idx = {c: i for i, c in enumerate(present_classes)}
valid_idx = [i for i, (t, p) in enumerate(zip(y_true_disease, y_pred_disease))
             if t in class_to_idx and p in class_to_idx]

if len(valid_idx) >= 10:
    Y_true = label_binarize(
        [class_to_idx[y_true_disease[i]] for i in valid_idx],
        classes=list(range(len(present_classes))),
    )
    Y_pred = np.zeros((len(valid_idx), len(present_classes)))
    for row, i in enumerate(valid_idx):
        Y_pred[row, class_to_idx[y_pred_disease[i]]] = 1.0

    roc_data = {}
    auc_scores = {}
    for i, cls in enumerate(present_classes):
        yt = Y_true[:, i]
        yp = Y_pred[:, i]
        if len(np.unique(yt)) < 2:
            continue
        fpr, tpr, _ = roc_curve(yt, yp)
        roc_data[cls] = (fpr, tpr, auc(fpr, tpr))
        auc_scores[cls] = roc_data[cls][2]

    macro_auc = np.mean(list(auc_scores.values())) if auc_scores else 0.0
    print(f"Macro AUC (one-vs-rest): {macro_auc:.4f}")
    print(f"Classes with AUC computed: {len(auc_scores)}")
else:
    macro_auc = 0.0
    roc_data = {}
    auc_scores = {}
    print("Insufficient valid samples for ROC computation")

# ------------------------------------------------------------
# Plots
# ------------------------------------------------------------
# Confusion matrix
fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.set_title(f"Confusion Matrix (n={N_EVAL}, acc={acc*100:.1f}%)", fontsize=14)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=6)
ax.set_yticklabels(labels, fontsize=6)
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=100, bbox_inches="tight")
plt.close()

# Per-class accuracy
fig, ax = plt.subplots(figsize=(14, max(6, len(per_class) * 0.3)))
classes_sorted = sorted(per_class.keys(), key=lambda c: per_class[c][2])
accs = [per_class[c][2] * 100 for c in classes_sorted]
colors_bar = ["#d62728" if a < 50 else "#ff7f0e" if a < 75 else "#2ca02c" for a in accs]
ax.barh(range(len(classes_sorted)), accs, color=colors_bar)
ax.set_yticks(range(len(classes_sorted)))
ax.set_yticklabels(classes_sorted, fontsize=8)
ax.set_xlabel("Accuracy (%)")
ax.set_title(f"Per-Class Disease Accuracy (Overall = {acc*100:.1f}%)")
ax.set_xlim(0, 100)
ax.axvline(x=acc*100, color="black", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "per_class_accuracy.png"), dpi=100, bbox_inches="tight")
plt.close()

# ROC curves
if roc_data:
    fig, ax = plt.subplots(figsize=(10, 8))
    top_curves = sorted(roc_data.items(), key=lambda x: x[1][2], reverse=True)[:20]
    colors_roc = plt.cm.tab20(np.linspace(0, 1, len(top_curves)))
    for i, (cls, (fpr, tpr, roc_auc)) in enumerate(top_curves):
        ax.plot(fpr, tpr, color=colors_roc[i], lw=1.5, label=f"{cls} (AUC={roc_auc:.2f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves — Top 20 Classes (Macro AUC = {macro_auc:.3f})")
    ax.legend(loc="lower right", fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "roc_curves.png"), dpi=100, bbox_inches="tight")
    plt.close()

# ------------------------------------------------------------
# Save all outputs
# ------------------------------------------------------------
metrics = {
    "evaluation": {
        "n_evaluated": N_EVAL,
        "max_new_tokens": MAX_NEW_TOKENS,
        "parse_failures": parse_failures,
    },
    "disease_metrics": {
        "accuracy": float(acc),
        "accuracy_excluding_parse_failures": float(real_acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
        "precision_weighted": float(p_weighted),
        "recall_weighted": float(r_weighted),
        "f1_weighted": float(f_weighted),
        "precision_micro": float(p_micro),
        "recall_micro": float(r_micro),
        "f1_micro": float(f_micro),
    },
    "plant_metrics": {
        "accuracy": float(plant_acc),
        "f1_macro": float(f_p),
    },
    "roc_auc": {
        "macro_auc": float(macro_auc),
        "per_class_auc": {k: float(v) for k, v in auc_scores.items()},
    },
    "per_class_accuracy": {k: float(v[2]) for k, v in per_class.items()},
    "per_class_support": {k: int(v[1]) for k, v in per_class.items()},
    "confusion_matrix": cm.tolist(),
    "class_labels": labels,
    "top_confusions": [
        {"true": t, "pred": p, "count": int(c)}
        for (t, p), c in confusions.most_common(30)
    ],
}
with open(os.path.join(OUT_DIR, "evaluation_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

with open(os.path.join(OUT_DIR, "raw_predictions.json"), "w") as f:
    json.dump(raw_outputs, f, indent=2)

with open(os.path.join(OUT_DIR, "classification_report.txt"), "w") as f:
    f.write(f"Overall Accuracy: {acc*100:.2f}%\n")
    f.write(f"Accuracy excluding parse failures: {real_acc*100:.2f}%\n")
    f.write(f"Macro F1: {f_macro*100:.2f}%\n")
    f.write(f"Weighted F1: {f_weighted*100:.2f}%\n")
    f.write(f"Macro AUC: {macro_auc:.4f}\n")
    f.write(f"Parse failures: {parse_failures}/{N_EVAL}\n\n")
    f.write(report)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"Dataset:                       {len(train_data)} train / {len(val_data)} val")
print(f"Classes evaluated:             {len(labels)}")
print(f"Parse failures (unknown):      {parse_failures}/{N_EVAL}")
print(f"Disease Accuracy:              {acc*100:.2f}%")
print(f"Disease Accuracy (no parse fails): {real_acc*100:.2f}%")
print(f"Disease Macro F1:              {f_macro*100:.2f}%")
print(f"Disease Weighted F1:           {f_weighted*100:.2f}%")
print(f"Plant Accuracy:                {plant_acc*100:.2f}%")
print(f"Macro AUC (OvR):               {macro_auc:.4f}")
print(f"Imbalance ratio:               {imbalance_ratio:.2f}")
print("=" * 70)
print(f"\nAll outputs saved to: {OUT_DIR}")
for fname in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, fname)
    print(f"  {fname:<40}  {os.path.getsize(fpath)/1024:>8.1f} KB")
print("=" * 70)

PLANT DISEASE MODEL — RE-EVALUATION (FIXED PARSER)
Evaluation samples: 300
Device: cuda:0
Max new tokens: 150

DATASET ANALYSIS

Training samples: 4104
Validation samples: 456
Unique classes (train): 21
Unique classes (val):   21

Class balance (training):
  Min samples/class: 103
  Max samples/class: 1294
  Mean samples/class: 195.4
  Std dev: 252.6
  Imbalance ratio (max/min): 12.56

RUNNING INFERENCE ON VALIDATION SET


Evaluating: 100%|██████████| 300/300 [28:50<00:00,  5.77s/it]



Parse failures (pred=unknown): 66 / 300

DISEASE-LEVEL METRICS

Overall Accuracy: 69.33%  (208/300)

Metric                    Macro   Weighted      Micro
----------------------------------------------------
Precision                70.62%     85.53%     69.33%
Recall                   51.33%     69.33%     69.33%
F1                       56.36%     73.78%     69.33%

PER-CLASS CLASSIFICATION REPORT
                                      precision    recall  f1-score   support

                          apple scab      0.800     0.500     0.615         8
                      bacterial spot      1.000     0.538     0.700        26
                           black rot      0.773     0.895     0.829        19
                    cedar apple rust      0.000     0.000     0.000         2
 cercospora leaf spot gray leaf spot      1.000     0.200     0.333        10
                         common rust      1.000     0.375     0.545         8
                        early blight      0.750  

In [89]:
# ============================================================
# PACKAGE EVALUATION IMAGES + PLOTS AS ZIP
# Creates a zip containing:
#   - All 300 validation images (renamed with GT and prediction)
#   - Confusion matrix, ROC curves, per-class accuracy, class distribution
#   - Raw predictions JSON
#   - Metrics JSON
# ============================================================

import os
import json
import zipfile
from pathlib import Path
from PIL import Image

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
EVAL_DIR = "/kaggle/working/evaluation_outputs_fixed"   # metrics outputs
IMG_OUT_DIR = "/kaggle/working/evaluation_images"       # image staging
ZIP_PATH = "/kaggle/working/krishinet-evaluation-bundle.zip"

os.makedirs(IMG_OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load raw predictions
# ------------------------------------------------------------
predictions_path = os.path.join(EVAL_DIR, "raw_predictions.json")
if not os.path.exists(predictions_path):
    raise FileNotFoundError(
        f"Predictions file not found at {predictions_path}. "
        "Run the fixed evaluation cell first."
    )

with open(predictions_path) as f:
    predictions = json.load(f)

print(f"Loaded {len(predictions)} predictions from {predictions_path}")

# ------------------------------------------------------------
# Export each validation image with descriptive filename
# ------------------------------------------------------------
# Filename format:  {idx:03d}__GT-{gt_disease}__PRED-{pred_disease}__{correct}.jpg
# Example: 000__GT-healthy__PRED-healthy__CORRECT.jpg
#          001__GT-early blight__PRED-late blight__WRONG.jpg

MAX_IMAGES = min(300, len(predictions))   # export up to 300 images

def sanitize(s: str) -> str:
    """Remove characters that are invalid in filenames."""
    for ch in ['/', '\\', ':', '*', '?', '"', '<', '>', '|', ',', '(', ')']:
        s = s.replace(ch, '')
    return s.strip().replace(' ', '_')

exported = 0
correct_count = 0
wrong_count = 0

print(f"\nExporting up to {MAX_IMAGES} images to {IMG_OUT_DIR}...")

for i in range(MAX_IMAGES):
    try:
        sample = val_data[i]
        img = sample["image"]

        pred_entry = predictions[i] if i < len(predictions) else None
        if pred_entry is None:
            continue

        gt = sanitize(pred_entry.get("gt_disease", "unknown"))
        pred = sanitize(pred_entry.get("pred_disease", "unknown"))
        correct = "CORRECT" if gt == pred else "WRONG"

        if gt == pred:
            correct_count += 1
        else:
            wrong_count += 1

        filename = f"{i:03d}__GT-{gt}__PRED-{pred}__{correct}.jpg"
        out_path = os.path.join(IMG_OUT_DIR, filename)

        # Convert to RGB (some images may be RGBA or palette mode)
        if img.mode != "RGB":
            img = img.convert("RGB")

        img.save(out_path, "JPEG", quality=85)
        exported += 1

    except Exception as e:
        print(f"  Failed to export sample {i}: {e}")

print(f"\nExported: {exported}")
print(f"  Correct: {correct_count}")
print(f"  Wrong:   {wrong_count}")

# ------------------------------------------------------------
# Copy metrics and plots into the same staging folder
# ------------------------------------------------------------
EXTRA_FILES = [
    "evaluation_metrics.json",
    "dataset_stats.json",
    "classification_report.txt",
    "confusion_matrix.png",
    "roc_curves.png",
    "per_class_accuracy.png",
    "class_distribution.png",
]

print("\nCopying metrics and plots...")
for fname in EXTRA_FILES:
    src = os.path.join(EVAL_DIR, fname)
    if os.path.exists(src):
        dst = os.path.join(IMG_OUT_DIR, fname)
        with open(src, "rb") as fin, open(dst, "wb") as fout:
            fout.write(fin.read())
        print(f"  Copied {fname}")
    else:
        print(f"  Skipped {fname} (not found)")

# ------------------------------------------------------------
# Write a README inside the zip
# ------------------------------------------------------------
readme_content = f"""# KrishiNet-VLM Evaluation Bundle

Generated from Kaggle notebook evaluation.

## Contents

### validation_images/
{exported} validation images, each named with its ground truth and prediction:
  Format: {{idx:03d}}__GT-{{ground_truth}}__PRED-{{prediction}}__{{CORRECT|WRONG}}.jpg

  Summary:
    Correct: {correct_count}
    Wrong:   {wrong_count}
    Total:   {exported}

### metrics/
- evaluation_metrics.json     Full metrics in machine-readable format
- dataset_stats.json          Dataset distribution statistics
- classification_report.txt   Per-class precision/recall/F1/support

### plots/
- confusion_matrix.png        Full confusion matrix heatmap
- roc_curves.png              ROC curves for top 20 classes
- per_class_accuracy.png      Per-class accuracy bar chart
- class_distribution.png      Training class distribution

## Model
- Base: HuggingFaceTB/SmolVLM-500M-Instruct
- Fine-tuned: QLoRA r=16 alpha=32, 2 epochs
- Training: 4,104 samples, 21 classes
- Validation: 456 samples

## Reported Metrics
- Disease accuracy:                  69.33%
- Disease accuracy (no parse fails): 88.89%
- Macro F1:                          56.36%
- Weighted F1:                       73.78%
- Plant accuracy:                    75.00%
- Macro AUC (one-vs-rest):           0.9317
"""

readme_path = os.path.join(IMG_OUT_DIR, "README.md")
with open(readme_path, "w") as f:
    f.write(readme_content)
print(f"\nWrote README.md")

# ------------------------------------------------------------
# Create the zip
# ------------------------------------------------------------
print(f"\nPackaging into {ZIP_PATH}...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(IMG_OUT_DIR):
        for fname in files:
            full_path = os.path.join(root, fname)
            rel_path = os.path.relpath(full_path, IMG_OUT_DIR)
            zf.write(full_path, rel_path)

zip_size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"\nZip created: {ZIP_PATH}")
print(f"Zip size: {zip_size_mb:.2f} MB")
print(f"Files in zip: {sum(1 for _ in zipfile.ZipFile(ZIP_PATH).namelist())}")

# ------------------------------------------------------------
# Generate a download link
# ------------------------------------------------------------
from IPython.display import FileLink, display

print("\n" + "=" * 70)
print("DOWNLOAD LINK")
print("=" * 70)
display(FileLink(ZIP_PATH, result_html_prefix="Click here to download: "))
print("=" * 70)
print("\nAlternative: Right sidebar → Output → find the .zip → Download")

Loaded 300 predictions from /kaggle/working/evaluation_outputs_fixed/raw_predictions.json

Exporting up to 300 images to /kaggle/working/evaluation_images...

Exported: 300
  Correct: 208
  Wrong:   92

Copying metrics and plots...
  Copied evaluation_metrics.json
  Copied dataset_stats.json
  Copied classification_report.txt
  Copied confusion_matrix.png
  Copied roc_curves.png
  Copied per_class_accuracy.png
  Skipped class_distribution.png (not found)

Wrote README.md

Packaging into /kaggle/working/krishinet-evaluation-bundle.zip...

Zip created: /kaggle/working/krishinet-evaluation-bundle.zip
Zip size: 6.57 MB
Files in zip: 307

DOWNLOAD LINK


/kaggle/working/krishinet-evaluation-bundle.zip


Alternative: Right sidebar → Output → find the .zip → Download


In [90]:
# ============================================================
# COMPLETE VISUALIZATION PACKAGE
# Generates histograms, pie charts, curves, confusion matrix,
# ROC, PR curves, and all metrics — packaged as a zip.
# ============================================================

import os
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
from IPython.display import FileLink, display

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
EVAL_DIR = "/kaggle/working/evaluation_outputs_fixed"
OUT_DIR = "/kaggle/working/visualization_package"
ZIP_PATH = "/kaggle/working/krishinet-visualization-package.zip"

os.makedirs(OUT_DIR, exist_ok=True)

# Matplotlib global style
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
})

print("=" * 70)
print("GENERATING COMPLETE VISUALIZATION PACKAGE")
print("=" * 70)

# ------------------------------------------------------------
# Load predictions and metrics
# ------------------------------------------------------------
with open(os.path.join(EVAL_DIR, "raw_predictions.json")) as f:
    predictions = json.load(f)

with open(os.path.join(EVAL_DIR, "evaluation_metrics.json")) as f:
    metrics = json.load(f)

with open(os.path.join(EVAL_DIR, "dataset_stats.json")) as f:
    dataset_stats = json.load(f)

y_true = [p["gt_disease"] for p in predictions]
y_pred = [p["pred_disease"] for p in predictions]

print(f"Loaded {len(predictions)} predictions")
print(f"Classes: {len(set(y_true))}")
print()

# ------------------------------------------------------------
# 1. HISTOGRAM: Per-class accuracy distribution
# ------------------------------------------------------------
print("[1/12] Per-class accuracy histogram")

per_class_acc = metrics["per_class_accuracy"]
accs = [v * 100 for v in per_class_acc.values()]

fig, ax = plt.subplots(figsize=(10, 6))
bins = np.arange(0, 110, 10)
n, bins_out, patches = ax.hist(accs, bins=bins, edgecolor="black", color="#4C72B0", alpha=0.75)
ax.set_xlabel("Per-class accuracy (%)")
ax.set_ylabel("Number of classes")
ax.set_title(f"Distribution of Per-Class Accuracy (n={len(accs)} classes)")
ax.set_xticks(bins)
ax.grid(axis="y", alpha=0.3)

# Annotate counts on bars
for count, x in zip(n, bins_out[:-1]):
    if count > 0:
        ax.text(x + 5, count + 0.15, str(int(count)), ha="center", fontsize=9)

# Add mean line
mean_acc = np.mean(accs)
ax.axvline(mean_acc, color="red", linestyle="--", linewidth=1.5,
           label=f"Mean: {mean_acc:.1f}%")
ax.legend()
plt.savefig(os.path.join(OUT_DIR, "01_per_class_accuracy_histogram.png"))
plt.close()

# ------------------------------------------------------------
# 2. HISTOGRAM: Training samples per class
# ------------------------------------------------------------
print("[2/12] Training samples histogram")

train_counts = dataset_stats["class_distribution"]
counts = list(train_counts.values())

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(counts, bins=20, edgecolor="black", color="#55A868", alpha=0.75)
ax.set_xlabel("Training samples per class")
ax.set_ylabel("Number of classes")
ax.set_title(f"Distribution of Training Samples per Class (n={len(counts)} classes)")
ax.axvline(np.mean(counts), color="red", linestyle="--", linewidth=1.5,
           label=f"Mean: {np.mean(counts):.0f}")
ax.axvline(np.median(counts), color="orange", linestyle=":", linewidth=1.5,
           label=f"Median: {np.median(counts):.0f}")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.savefig(os.path.join(OUT_DIR, "02_training_samples_histogram.png"))
plt.close()

# ------------------------------------------------------------
# 3. PIE CHART: Correct vs Wrong vs Unknown
# ------------------------------------------------------------
print("[3/12] Prediction outcome pie chart")

total = len(y_true)
correct = sum(1 for t, p in zip(y_true, y_pred) if t == p and p != "unknown")
wrong = sum(1 for t, p in zip(y_true, y_pred) if t != p and p != "unknown")
unknown = sum(1 for p in y_pred if p == "unknown")

fig, ax = plt.subplots(figsize=(9, 9))
sizes = [correct, wrong, unknown]
labels = [f"Correct\n{correct} ({correct/total*100:.1f}%)",
          f"Wrong class\n{wrong} ({wrong/total*100:.1f}%)",
          f"Parse failure\n{unknown} ({unknown/total*100:.1f}%)"]
colors = ["#2ca02c", "#d62728", "#ff7f0e"]
explode = (0.05, 0.05, 0.1)

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, explode=explode,
    autopct="%1.1f%%", startangle=90, shadow=True,
    textprops={"fontsize": 11},
)
for t in autotexts:
    t.set_fontweight("bold")
ax.set_title(f"Prediction Outcomes (n={total})", fontsize=13, pad=20)
plt.savefig(os.path.join(OUT_DIR, "03_outcome_pie_chart.png"))
plt.close()

# ------------------------------------------------------------
# 4. PIE CHART: Top-10 class distribution in validation set
# ------------------------------------------------------------
print("[4/12] Validation class distribution pie chart")

val_counts = Counter(y_true)
top_10 = val_counts.most_common(10)
other = sum(v for k, v in val_counts.items() if k not in dict(top_10))

labels_pie = [k for k, v in top_10] + ["other"]
sizes_pie = [v for k, v in top_10] + [other]

fig, ax = plt.subplots(figsize=(11, 9))
colors_pie = plt.cm.tab20(np.linspace(0, 1, len(labels_pie)))
wedges, texts, autotexts = ax.pie(
    sizes_pie, labels=labels_pie, colors=colors_pie,
    autopct="%1.1f%%", startangle=90, textprops={"fontsize": 9},
)
for t in autotexts:
    t.set_fontsize(8)
ax.set_title(f"Validation Set Class Distribution (Top 10 of {len(val_counts)})",
             fontsize=13, pad=20)
plt.savefig(os.path.join(OUT_DIR, "04_validation_class_pie.png"))
plt.close()

# ------------------------------------------------------------
# 5. BAR CHART: Per-class accuracy (ascending, colored)
# ------------------------------------------------------------
print("[5/12] Per-class accuracy bar chart")

classes_sorted = sorted(per_class_acc.keys(), key=lambda c: per_class_acc[c])
accs_sorted = [per_class_acc[c] * 100 for c in classes_sorted]
colors_bar = ["#d62728" if a < 50 else "#ff7f0e" if a < 75 else "#2ca02c"
              for a in accs_sorted]

fig, ax = plt.subplots(figsize=(12, max(7, len(classes_sorted) * 0.35)))
bars = ax.barh(range(len(classes_sorted)), accs_sorted, color=colors_bar, edgecolor="black")
ax.set_yticks(range(len(classes_sorted)))
ax.set_yticklabels(classes_sorted, fontsize=9)
ax.set_xlabel("Accuracy (%)")
ax.set_title(f"Per-Class Disease Accuracy (Overall = {metrics['disease_metrics']['accuracy']*100:.1f}%)")
ax.set_xlim(0, 105)

# Annotate each bar with value and support
supports = metrics["per_class_support"]
for i, (cls, a) in enumerate(zip(classes_sorted, accs_sorted)):
    ax.text(a + 1, i, f"{a:.1f}%  (n={supports[cls]})", va="center", fontsize=8)

ax.axvline(x=metrics["disease_metrics"]["accuracy"]*100, color="black",
           linestyle="--", linewidth=1.5,
           label=f"Overall: {metrics['disease_metrics']['accuracy']*100:.1f}%")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.savefig(os.path.join(OUT_DIR, "05_per_class_accuracy_bar.png"))
plt.close()

# ------------------------------------------------------------
# 6. BAR CHART: Precision, Recall, F1 per class
# ------------------------------------------------------------
print("[6/12] Precision/Recall/F1 per class")

p_per, r_per, f_per, sup_per = precision_recall_fscore_support(
    y_true, y_pred,
    labels=classes_sorted,
    zero_division=0,
)

x = np.arange(len(classes_sorted))
width = 0.27

fig, ax = plt.subplots(figsize=(14, max(7, len(classes_sorted) * 0.35)))
ax.barh(x - width, p_per * 100, width, label="Precision", color="#4C72B0")
ax.barh(x, r_per * 100, width, label="Recall", color="#DD8452")
ax.barh(x + width, f_per * 100, width, label="F1", color="#55A868")
ax.set_yticks(x)
ax.set_yticklabels(classes_sorted, fontsize=9)
ax.set_xlabel("Score (%)")
ax.set_title("Per-Class Precision / Recall / F1")
ax.set_xlim(0, 105)
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.savefig(os.path.join(OUT_DIR, "06_precision_recall_f1_bar.png"))
plt.close()

# ------------------------------------------------------------
# 7. ROC CURVES (one-vs-rest)
# ------------------------------------------------------------
print("[7/12] ROC curves")

present_classes = sorted(set(y_true) - {"unknown"})
class_to_idx = {c: i for i, c in enumerate(present_classes)}
valid_idx = [i for i, (t, p) in enumerate(zip(y_true, y_pred))
             if t in class_to_idx and p in class_to_idx]

if len(valid_idx) >= 10:
    Y_true = label_binarize(
        [class_to_idx[y_true[i]] for i in valid_idx],
        classes=list(range(len(present_classes))),
    )
    Y_pred = np.zeros((len(valid_idx), len(present_classes)))
    for row, i in enumerate(valid_idx):
        Y_pred[row, class_to_idx[y_pred[i]]] = 1.0

    roc_data = {}
    for i, cls in enumerate(present_classes):
        yt = Y_true[:, i]
        yp = Y_pred[:, i]
        if len(np.unique(yt)) < 2:
            continue
        fpr, tpr, _ = roc_curve(yt, yp)
        roc_data[cls] = (fpr, tpr, auc(fpr, tpr))

    # Plot top 15
    fig, ax = plt.subplots(figsize=(10, 8))
    top = sorted(roc_data.items(), key=lambda x: x[1][2], reverse=True)[:15]
    colors_roc = plt.cm.tab20(np.linspace(0, 1, len(top)))
    for i, (cls, (fpr, tpr, roc_auc)) in enumerate(top):
        ax.plot(fpr, tpr, color=colors_roc[i], lw=1.8,
                label=f"{cls} (AUC={roc_auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    macro_auc = np.mean([v[2] for v in roc_data.values()])
    ax.set_title(f"ROC Curves — Top 15 Classes (Macro AUC = {macro_auc:.4f})")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(alpha=0.3)
    plt.savefig(os.path.join(OUT_DIR, "07_roc_curves.png"))
    plt.close()
    print(f"  Macro AUC: {macro_auc:.4f}")
else:
    macro_auc = 0.0
    roc_data = {}

# ------------------------------------------------------------
# 8. PRECISION-RECALL CURVES
# ------------------------------------------------------------
print("[8/12] Precision-Recall curves")

if len(valid_idx) >= 10:
    fig, ax = plt.subplots(figsize=(10, 8))
    top_pr = sorted(present_classes)[:15]
    colors_pr = plt.cm.tab20(np.linspace(0, 1, len(top_pr)))

    for i, cls in enumerate(top_pr):
        cls_i = class_to_idx.get(cls)
        if cls_i is None:
            continue
        yt = Y_true[:, cls_i]
        yp = Y_pred[:, cls_i]
        if len(np.unique(yt)) < 2:
            continue
        precision, recall, _ = precision_recall_curve(yt, yp)
        ap = average_precision_score(yt, yp)
        ax.plot(recall, precision, color=colors_pr[i], lw=1.8,
                label=f"{cls} (AP={ap:.3f})")

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curves — Top 15 Classes")
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.legend(loc="lower left", fontsize=8)
    ax.grid(alpha=0.3)
    plt.savefig(os.path.join(OUT_DIR, "08_precision_recall_curves.png"))
    plt.close()

# ------------------------------------------------------------
# 9. CONFUSION MATRIX (normalized)
# ------------------------------------------------------------
print("[9/12] Normalized confusion matrix")

all_labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=all_labels)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
ax.set_title(f"Normalized Confusion Matrix (n={len(y_true)})", fontsize=13)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(np.arange(len(all_labels)))
ax.set_yticks(np.arange(len(all_labels)))
ax.set_xticklabels(all_labels, rotation=90, fontsize=7)
ax.set_yticklabels(all_labels, fontsize=7)
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")

# Annotate cells with values > 0.1
for i in range(len(all_labels)):
    for j in range(len(all_labels)):
        if cm_norm[i, j] > 0.1:
            ax.text(j, i, f"{cm_norm[i, j]:.2f}",
                    ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black",
                    fontsize=6)
plt.savefig(os.path.join(OUT_DIR, "09_confusion_matrix_normalized.png"))
plt.close()

# ------------------------------------------------------------
# 10. METRICS SUMMARY BAR CHART
# ------------------------------------------------------------
print("[10/12] Overall metrics summary")

dm = metrics["disease_metrics"]
pm = metrics["plant_metrics"]
roc = metrics["roc_auc"]

metric_names = [
    "Disease\nAccuracy",
    "Disease\nPrecision (M)",
    "Disease\nRecall (M)",
    "Disease\nF1 (M)",
    "Disease\nF1 (W)",
    "Plant\nAccuracy",
    "Plant\nF1 (M)",
    "Macro\nAUC",
]
metric_values = [
    dm["accuracy"] * 100,
    dm["precision_macro"] * 100,
    dm["recall_macro"] * 100,
    dm["f1_macro"] * 100,
    dm["f1_weighted"] * 100,
    pm["accuracy"] * 100,
    pm["f1_macro"] * 100,
    roc["macro_auc"] * 100,
]

fig, ax = plt.subplots(figsize=(12, 7))
colors_metrics = ["#4C72B0" if v < 70 else "#DD8452" if v < 85 else "#55A868"
                  for v in metric_values]
bars = ax.bar(metric_names, metric_values, color=colors_metrics, edgecolor="black")
ax.set_ylabel("Score (%)")
ax.set_title("Overall Model Performance Summary")
ax.set_ylim(0, 105)
ax.axhline(y=70, color="gray", linestyle=":", alpha=0.5)
ax.axhline(y=85, color="gray", linestyle=":", alpha=0.5)
ax.grid(axis="y", alpha=0.3)

for bar, val in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 1.5,
            f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")

plt.xticks(rotation=0, fontsize=9)
plt.savefig(os.path.join(OUT_DIR, "10_overall_metrics_bar.png"))
plt.close()

# ------------------------------------------------------------
# 11. CORRECT vs WRONG per class (stacked bar)
# ------------------------------------------------------------
print("[11/12] Correct vs wrong per class")

support_per_class = metrics["per_class_support"]
correct_per_class = {}
wrong_per_class = {}
unknown_per_class = {}

for cls in classes_sorted:
    total_cls = support_per_class[cls]
    acc_cls = per_class_acc[cls]
    correct_cls = int(round(acc_cls * total_cls))
    correct_per_class[cls] = correct_cls
    # Remaining split between wrong-class and unknown
    remaining = total_cls - correct_cls
    unknown_cls = sum(1 for t, p in zip(y_true, y_pred)
                      if t == cls and p == "unknown")
    unknown_per_class[cls] = unknown_cls
    wrong_per_class[cls] = remaining - unknown_cls

fig, ax = plt.subplots(figsize=(12, max(7, len(classes_sorted) * 0.4)))
y_pos = np.arange(len(classes_sorted))
h = 0.7
correct_vals = [correct_per_class[c] for c in classes_sorted]
wrong_vals = [wrong_per_class[c] for c in classes_sorted]
unknown_vals = [unknown_per_class[c] for c in classes_sorted]

ax.barh(y_pos, correct_vals, h, label="Correct", color="#2ca02c")
ax.barh(y_pos, wrong_vals, h, left=correct_vals, label="Wrong class", color="#d62728")
ax.barh(y_pos, unknown_vals, h,
        left=[c + w for c, w in zip(correct_vals, wrong_vals)],
        label="Parse failure", color="#ff7f0e")

ax.set_yticks(y_pos)
ax.set_yticklabels(classes_sorted, fontsize=9)
ax.set_xlabel("Number of samples")
ax.set_title("Per-Class Prediction Breakdown (Correct / Wrong / Parse Failure)")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.savefig(os.path.join(OUT_DIR, "11_stacked_correct_wrong.png"))
plt.close()

# ------------------------------------------------------------
# 12. CUMULATIVE ACCURACY (how many classes cover X% of samples)
# ------------------------------------------------------------
print("[12/12] Cumulative class coverage")

val_counts_sorted = sorted(val_counts.values(), reverse=True)
cumsum = np.cumsum(val_counts_sorted) / sum(val_counts_sorted) * 100

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(cumsum) + 1), cumsum, marker="o", linewidth=2, color="#4C72B0")
ax.axhline(y=80, color="red", linestyle="--", alpha=0.5, label="80% coverage")
ax.axhline(y=95, color="orange", linestyle="--", alpha=0.5, label="95% coverage")

# Find how many classes cover 80% and 95%
n_80 = next((i + 1 for i, v in enumerate(cumsum) if v >= 80), len(cumsum))
n_95 = next((i + 1 for i, v in enumerate(cumsum) if v >= 95), len(cumsum))
ax.axvline(x=n_80, color="red", linestyle=":", alpha=0.5)
ax.axvline(x=n_95, color="orange", linestyle=":", alpha=0.5)
ax.text(n_80 + 0.2, 40, f"{n_80} classes\ncover 80%", color="red", fontsize=9)
ax.text(n_95 + 0.2, 60, f"{n_95} classes\ncover 95%", color="orange", fontsize=9)

ax.set_xlabel("Number of classes (sorted by frequency)")
ax.set_ylabel("Cumulative % of validation samples")
ax.set_title(f"Class Coverage Curve (total {len(val_counts)} classes)")
ax.set_xlim(0, len(cumsum) + 1)
ax.set_ylim(0, 105)
ax.grid(alpha=0.3)
ax.legend()
plt.savefig(os.path.join(OUT_DIR, "12_cumulative_coverage.png"))
plt.close()

# ------------------------------------------------------------
# 13. Export metrics as CSV
# ------------------------------------------------------------
import csv

csv_path = os.path.join(OUT_DIR, "per_class_metrics.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "class", "accuracy", "precision", "recall", "f1", "support"
    ])
    for cls in classes_sorted:
        writer.writerow([
            cls,
            f"{per_class_acc[cls]:.4f}",
            f"{p_per[classes_sorted.index(cls)]:.4f}",
            f"{r_per[classes_sorted.index(cls)]:.4f}",
            f"{f_per[classes_sorted.index(cls)]:.4f}",
            support_per_class[cls],
        ])

# Summary CSV
summary_csv = os.path.join(OUT_DIR, "overall_metrics.csv")
with open(summary_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerow(["disease_accuracy", f"{dm['accuracy']:.4f}"])
    writer.writerow(["disease_precision_macro", f"{dm['precision_macro']:.4f}"])
    writer.writerow(["disease_recall_macro", f"{dm['recall_macro']:.4f}"])
    writer.writerow(["disease_f1_macro", f"{dm['f1_macro']:.4f}"])
    writer.writerow(["disease_f1_weighted", f"{dm['f1_weighted']:.4f}"])
    writer.writerow(["plant_accuracy", f"{pm['accuracy']:.4f}"])
    writer.writerow(["plant_f1_macro", f"{pm['f1_macro']:.4f}"])
    writer.writerow(["macro_auc", f"{roc['macro_auc']:.4f}"])
    writer.writerow(["n_evaluated", metrics["evaluation"]["n_evaluated"]])
    writer.writerow(["parse_failures", metrics["evaluation"].get("parse_failures", "N/A")])

print(f"\nWrote {csv_path}")
print(f"Wrote {summary_csv}")

# ------------------------------------------------------------
# Package everything into a zip
# ------------------------------------------------------------
print("\nPackaging into zip...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(OUT_DIR)):
        full_path = os.path.join(OUT_DIR, fname)
        zf.write(full_path, fname)

zip_size = os.path.getsize(ZIP_PATH) / 1e6
print(f"\nZIP created: {ZIP_PATH}  ({zip_size:.2f} MB)")
print(f"Files in zip: {len(os.listdir(OUT_DIR))}")

# ------------------------------------------------------------
# Generate download link
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("DOWNLOAD LINK")
print("=" * 70)
display(FileLink(ZIP_PATH, result_html_prefix="Click to download: "))
print("=" * 70)

# ------------------------------------------------------------
# Final listing
# ------------------------------------------------------------
print("\nFiles generated:")
for fname in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, fname)
    print(f"  {fname:<50}  {os.path.getsize(fpath)/1024:>8.1f} KB")

GENERATING COMPLETE VISUALIZATION PACKAGE
Loaded 300 predictions
Classes: 21

[1/12] Per-class accuracy histogram
[2/12] Training samples histogram
[3/12] Prediction outcome pie chart
[4/12] Validation class distribution pie chart
[5/12] Per-class accuracy bar chart
[6/12] Precision/Recall/F1 per class
[7/12] ROC curves
  Macro AUC: 0.9317
[8/12] Precision-Recall curves
[9/12] Normalized confusion matrix


/tmp/ipykernel_49/4043859215.py:325: RuntimeWarning: invalid value encountered in divide
  cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)


[10/12] Overall metrics summary
[11/12] Correct vs wrong per class
[12/12] Cumulative class coverage

Wrote /kaggle/working/visualization_package/per_class_metrics.csv
Wrote /kaggle/working/visualization_package/overall_metrics.csv

Packaging into zip...

ZIP created: /kaggle/working/krishinet-visualization-package.zip  (1.19 MB)
Files in zip: 14

DOWNLOAD LINK


/kaggle/working/krishinet-visualization-package.zip


Files generated:
  01_per_class_accuracy_histogram.png                     48.8 KB
  02_training_samples_histogram.png                       48.8 KB
  03_outcome_pie_chart.png                                92.2 KB
  04_validation_class_pie.png                            111.0 KB
  05_per_class_accuracy_bar.png                          158.0 KB
  06_precision_recall_f1_bar.png                         100.9 KB
  07_roc_curves.png                                      167.7 KB
  08_precision_recall_curves.png                         197.4 KB
  09_confusion_matrix_normalized.png                     187.7 KB
  10_overall_metrics_bar.png                              56.3 KB
  11_stacked_correct_wrong.png                           109.3 KB
  12_cumulative_coverage.png                              82.8 KB
  overall_metrics.csv                                      0.2 KB
  per_class_metrics.csv                                    1.1 KB
